In [3]:
from pathlib import Path
import hashlib
import importlib.metadata as metadata
import shutil
import subprocess
import sys

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
CHECKPOINT = ROOT.parent / "models/yolov8s.pt"
EXPECTED_SHA = "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"

def run_readonly(args, cwd=None):
    try:
        result = subprocess.run(
            args, cwd=cwd, capture_output=True, text=True, timeout=30
        )
        return {
            "returncode": result.returncode,
            "stdout": result.stdout.strip(),
            "stderr": result.stderr.strip(),
        }
    except Exception as exc:
        return {"error": str(exc)}

print("Python:", sys.version.split()[0])
print("Repository exists:", ROOT.is_dir())

for package in ("torch", "torchvision", "ultralytics", "numpy", "pycocotools"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nGPU:", run_readonly([
    "nvidia-smi",
    "--query-gpu=name,memory.total,memory.free",
    "--format=csv,noheader",
]))

if ROOT.is_dir():
    print("\nGit HEAD:", run_readonly(["git", "rev-parse", "HEAD"], ROOT))
    print("Git status:", run_readonly(
        ["git", "--no-optional-locks", "status", "--short"], ROOT
    ))
    print("Disk free GiB:", round(shutil.disk_usage(ROOT).free / 2**30, 2))

    paths = [
        "src/baseline05a_runner_v1.py",
        "src/fault05b_validation_v1.py",
        "src/screen05b_campaign_v1.py",
        "src/contextual06a_runner_v1.py",
        "experiments/notebook05b/int8_screening_plan_v1.json",
        "experiments/notebook06a/masked_bit_feasibility_protocol_v2.json",
    ]
    print("\nRequired files:")
    for relative in paths:
        path = ROOT / relative
        print(relative, "FOUND" if path.is_file() else "MISSING")

print("\nCheckpoint exists:", CHECKPOINT.is_file())
if CHECKPOINT.is_file():
    digest = hashlib.sha256()
    with CHECKPOINT.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    actual_sha = digest.hexdigest()
    print("Checkpoint SHA256:", actual_sha)
    print("Checkpoint identity:", "PASS" if actual_sha == EXPECTED_SHA else "FAIL")

Python: 3.13.16
Repository exists: True
torch: 2.14.1+cu130
torchvision: 0.29.1+cu130
ultralytics: NOT INSTALLED
numpy: 2.4.6
pycocotools: 2.0.11

GPU: {'returncode': 0, 'stdout': 'Tesla T4, 15360 MiB, 14913 MiB', 'stderr': ''}

Git HEAD: {'returncode': 0, 'stdout': '97aca930e3c9fdb47e714e9d49decf4778d894c0', 'stderr': ''}
Git status: {'error': "Command '['git', '--no-optional-locks', 'status', '--short']' timed out after 30 seconds"}
Disk free GiB: 7.34

Required files:
src/baseline05a_runner_v1.py FOUND
src/fault05b_validation_v1.py FOUND
src/screen05b_campaign_v1.py FOUND
src/contextual06a_runner_v1.py FOUND
experiments/notebook05b/int8_screening_plan_v1.json FOUND
experiments/notebook06a/masked_bit_feasibility_protocol_v2.json FOUND

Checkpoint exists: True
Checkpoint SHA256: 1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a
Checkpoint identity: PASS


In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
print("Project directory exists:", project.is_dir())
print("Repository directory exists:", (project / "code").is_dir())
print("Checkpoint exists:", (project / "models/yolov8s.pt").is_file())

Mounted at /content/drive
Project directory exists: True
Repository directory exists: True
Checkpoint exists: True


In [4]:
from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
protocol_path = (
    ROOT / "experiments/notebook06a/masked_bit_feasibility_protocol_v2.json"
)

with protocol_path.open("r", encoding="utf-8") as stream:
    protocol = json.load(stream)

print("Protocol:", protocol_path.relative_to(ROOT))
print(json.dumps(protocol, indent=2, ensure_ascii=False))

Protocol: experiments/notebook06a/masked_bit_feasibility_protocol_v2.json
{
  "status": "FROZEN_INITIAL_FEASIBILITY_PROTOCOL",
  "source_commit": "710335d2f526899ab1b27c1f34c80660b2a3b199",
  "seed": 20261116,
  "representation": "05a_shared_logical_INT8_per_output_channel",
  "scope": "63_eligible_Conv2d_weight_tensors",
  "task": "predict_one_masked_clean_bit_at_a_known_weight_and_bit_position",
  "bits": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7
  ],
  "split": {
    "unit": "output_channel",
    "within_each_layer": "seeded_60_20_20_train_calibration_test",
    "channel_groups_disjoint": true,
    "all_context_weights_must_remain_in_the_target_output_channel": true,
    "maximum_target_weights_per_layer": {
      "train": 512,
      "calibration": 256,
      "test": 256
    },
    "sampling": "uniform_without_replacement_within_each_partition"
  },
  "comparisons": {
    "A": "smoothed_bit_frequency_by_layer_and_bit",
    "B": "smoothed_conditional_frequency_given_ot

In [5]:
from pathlib import Path

module_path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "src/contextual06a_runner_v1.py"
)

lines = module_path.read_text(encoding="utf-8").splitlines()
print(f"Module: {module_path.name} | Total lines: {len(lines)}")

for number, line in enumerate(lines, start=1):
    print(f"{number:04d}: {line}")

Module: contextual06a_runner_v1.py | Total lines: 421
0001: """Fixed-protocol masked-bit feasibility comparison, not a fault repair system.
0002: 
0003: No Torch, CUDA, model inference, images, network requests or Git operations.
0004: No automatic hyperparameter selection or use of test labels during fitting.
0005: """
0006: import contextlib
0007: import hashlib
0008: import io
0009: import json
0010: import math
0011: import os
0012: import time
0013: from pathlib import Path
0014: 
0015: PROTOCOL_SHA = 'd851b7a91569ffa155d3e85ffd7afa1fb80e0df851857979cde7a51e392bceec'
0016: DATASET_SHA = '6e2a4c366321b392f59a330b2cc2810a965897feb9d72e34e98b492666a1df0e'
0017: MANIFEST_SHA = '515632610a2503bce69fd99b21391d37968d217457c54f39d56f73143ea307fe'
0018: METHODS = ('A','B','C','D')
0019: PARTS = ('train','calibration','test')
0020: CLIP = 1e-6
0021: BOOTSTRAP_SEED = 20261117
0022: BOOTSTRAP_DRAWS = 2000
0023: 
0024: 
0025: def canonical(value):
0026:     return json.dumps(value, sort_keys=T

In [6]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

print("Dependency versions:")
for package in ("numpy", "scikit-learn", "scipy", "joblib", "threadpoolctl"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

directories = [
    ROOT / "results/notebook06a/masked_bit_dataset_v1",
    ROOT / "results/notebook06a/masked_bit_comparison_v1",
    ROOT / "docs/research_reports/notebook06a_feasibility_v1",
]

for directory in directories:
    print(f"\nDirectory: {directory.relative_to(ROOT)}")
    if not directory.is_dir():
        print("MISSING")
        continue

    files = sorted(path for path in directory.iterdir() if path.is_file())
    print(f"Files: {len(files)}")
    for path in files:
        print(f"  {path.name} | {path.stat().st_size:,} bytes")

Dependency versions:
numpy: 2.4.6
scikit-learn: 1.6.1
scipy: 1.16.3
joblib: 1.6.0
threadpoolctl: 3.7.0

Directory: results/notebook06a/masked_bit_dataset_v1
Files: 2
  dataset.npz | 1,398,985 bytes
  manifest.json | 169,997 bytes

Directory: results/notebook06a/masked_bit_comparison_v1
Files: 26
  bit_0_models.joblib | 150,339 bytes
  bit_0_probabilities.npz | 298,673 bytes
  bit_0_result.json | 3,265 bytes
  bit_1_models.joblib | 150,367 bytes
  bit_1_probabilities.npz | 306,866 bytes
  bit_1_result.json | 3,261 bytes
  bit_2_models.joblib | 153,628 bytes
  bit_2_probabilities.npz | 339,067 bytes
  bit_2_result.json | 3,264 bytes
  bit_3_models.joblib | 153,828 bytes
  bit_3_probabilities.npz | 313,552 bytes
  bit_3_result.json | 3,273 bytes
  bit_4_models.joblib | 154,049 bytes
  bit_4_probabilities.npz | 334,085 bytes
  bit_4_result.json | 3,262 bytes
  bit_5_models.joblib | 155,062 bytes
  bit_5_probabilities.npz | 340,749 bytes
  bit_5_result.json | 3,270 bytes
  bit_6_models.jobl

In [7]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
SOURCE = ROOT / "results/notebook06a/masked_bit_dataset_v1"
RESULTS = ROOT / "results/notebook06a/masked_bit_comparison_v1"

def sha256_bytes(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def check_file(path, expected):
    actual = sha256_bytes(path.read_bytes())
    if actual != expected:
        raise RuntimeError(f"SHA256 mismatch: {path.name}")
    print(f"PASS: {path.name}")

def read_record(path):
    envelope = json.loads(path.read_bytes())
    payload = envelope["payload"]
    if sha256_bytes(canonical(payload)) != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload checksum mismatch: {path.name}")
    return payload

session = read_record(RESULTS / "session.json")
fingerprint = sha256_bytes(canonical(session))

check_file(
    ROOT / "src/contextual06a_runner_v1.py",
    session["source_sha256"]
)
check_file(
    ROOT / "experiments/notebook06a/masked_bit_feasibility_protocol_v2.json",
    session["protocol_sha256"]
)
check_file(SOURCE / "manifest.json", session["manifest_sha256"])
check_file(SOURCE / "dataset.npz", session["dataset_sha256"])

manifest = json.loads((SOURCE / "manifest.json").read_bytes())
if (
    manifest["protocol_sha256"] != session["protocol_sha256"]
    or manifest["dataset_sha256"] != session["dataset_sha256"]
):
    raise RuntimeError("Manifest provenance mismatch")

for bit in range(8):
    record = read_record(RESULTS / f"bit_{bit}_result.json")
    if record["bit"] != bit or record["session_fingerprint"] != fingerprint:
        raise RuntimeError(f"Session mismatch: bit {bit}")
    check_file(
        RESULTS / f"bit_{bit}_models.joblib",
        record["models_sha256"]
    )
    check_file(
        RESULTS / f"bit_{bit}_probabilities.npz",
        record["prediction_sha256"]
    )

print("\nRecorded environment:")
print(json.dumps(session["environment"], indent=2))
print("\nPartition counts:")
print(json.dumps(manifest["partition_counts"], indent=2))
print("\nPASS: artifact checksums and session links are consistent.")
print("Models were not loaded; metrics were not recomputed.")

PASS: contextual06a_runner_v1.py
PASS: masked_bit_feasibility_protocol_v2.json
PASS: manifest.json
PASS: dataset.npz
PASS: bit_0_models.joblib
PASS: bit_0_probabilities.npz
PASS: bit_1_models.joblib
PASS: bit_1_probabilities.npz
PASS: bit_2_models.joblib
PASS: bit_2_probabilities.npz
PASS: bit_3_models.joblib
PASS: bit_3_probabilities.npz
PASS: bit_4_models.joblib
PASS: bit_4_probabilities.npz
PASS: bit_5_models.joblib
PASS: bit_5_probabilities.npz
PASS: bit_6_models.joblib
PASS: bit_6_probabilities.npz
PASS: bit_7_models.joblib
PASS: bit_7_probabilities.npz

Recorded environment:
{
  "python": "3.13.18",
  "packages": {
    "numpy": "2.4.6",
    "scikit-learn": "1.6.1",
    "scipy": "1.16.3",
    "joblib": "1.6.0",
    "threadpoolctl": "3.7.0"
  },
  "thread_limit": 1
}

Partition counts:
{
  "train": 32256,
  "calibration": 16034,
  "test": 16061
}

PASS: artifact checksums and session links are consistent.
Models were not loaded; metrics were not recomputed.


In [8]:
from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "results/notebook06a/masked_bit_dataset_v1/manifest.json"
manifest = json.loads(path.read_bytes())

print("Manifest fields:", sorted(manifest))
print("\nFirst layer inventory:")
print(json.dumps(manifest["inventory"][0], indent=2, ensure_ascii=False))

print("\nChannel coverage:")
totals = {"train": 0, "calibration": 0, "test": 0}

for row in manifest["inventory"]:
    groups = {part: set(row["channels"][part]) for part in totals}
    if any(
        groups[a] & groups[b]
        for a, b in (
            ("train", "calibration"),
            ("train", "test"),
            ("calibration", "test"),
        )
    ):
        raise RuntimeError(f"Overlapping channel groups: {row['layer']}")

    counts = {part: len(groups[part]) for part in totals}
    for part, count in counts.items():
        totals[part] += count

    print(
        f"{row['layer']} | "
        f"train={counts['train']} "
        f"calibration={counts['calibration']} "
        f"test={counts['test']}"
    )

print("\nTotal layer-channel groups:", totals)
print("PASS: declared channel groups are disjoint.")

Manifest fields: ['channel_rounding', 'context_columns', 'dataset_sha256', 'feature_rules', 'inventory', 'numpy_version', 'partition_counts', 'protocol_sha256', 'sampling', 'status']

First layer inventory:
{
  "layer_id": 0,
  "layer": "model.0.conv",
  "integer_array_sha256": "93706b16ecee29c1020b2cbc346b8d1c1894c8da9f4d091a16be1ce5a0ce5858",
  "channels": {
    "train": [
      17,
      21,
      7,
      8,
      10,
      1,
      19,
      0,
      9,
      14,
      31,
      25,
      30,
      15,
      13,
      26,
      4,
      3,
      23
    ],
    "calibration": [
      11,
      20,
      22,
      28,
      12,
      24
    ],
    "test": [
      16,
      18,
      2,
      6,
      27,
      5,
      29
    ]
  }
}

Channel coverage:
model.0.conv | train=19 calibration=6 test=7
model.1.conv | train=38 calibration=12 test=14
model.2.cv1.conv | train=38 calibration=12 test=14
model.2.cv2.conv | train=38 calibration=12 test=14
model.2.m.0.cv1.conv | train=19 calibrati

In [9]:
from pathlib import Path
import json

path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "results/notebook06a/masked_bit_dataset_v1/manifest.json"
)
manifest = json.loads(path.read_bytes())

for field in (
    "status",
    "context_columns",
    "feature_rules",
    "sampling",
    "channel_rounding",
):
    print(f"\n{field}:")
    print(json.dumps(manifest[field], indent=2, ensure_ascii=False))


status:
"DATA_PREPARED_NO_PREDICTOR_TRAINED"

context_columns:
[
  "mean",
  "standard_deviation",
  "mean_absolute",
  "zero_fraction",
  "bit_0_frequency",
  "bit_1_frequency",
  "bit_2_frequency",
  "bit_3_frequency",
  "bit_4_frequency",
  "bit_5_frequency",
  "bit_6_frequency",
  "bit_7_frequency",
  "neighbor_count"
]

feature_rules:
[
  "For bit b, exclude column b from target bits before prediction.",
  "Channel and flat_index are audit metadata, never predictor inputs.",
  "Context excludes the target and stays inside its output channel."
]

sampling:
"PCG64; per-layer seed from first 16 hex digits of SHA256([seed, layer])"

channel_rounding:
"floor_0.6_train_floor_0.2_calibration_remainder_test"


In [10]:
from pathlib import Path
import importlib.util
import json
import numpy as np

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
SOURCE = ROOT / "results/notebook06a/masked_bit_dataset_v1"
module_path = ROOT / "src/contextual06a_runner_v1.py"

# Import definitions only; do not call the experiment runner.
spec = importlib.util.spec_from_file_location("contextual06a_audit", module_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

manifest = json.loads((SOURCE / "manifest.json").read_bytes())

with np.load(SOURCE / "dataset.npz", allow_pickle=False) as archive:
    expected = {
        f"{part}_{key}"
        for part in module.PARTS
        for key in ("bits", "context", "layer", "channel", "flat_index")
    }
    if set(archive.files) != expected:
        raise RuntimeError("Unexpected dataset entries.")

    data06a = {
        part: {
            key: archive[f"{part}_{key}"].copy()
            for key in ("bits", "context", "layer", "channel", "flat_index")
        }
        for part in module.PARTS
    }

module.validate_dataset(data06a, manifest)

for part, arrays in data06a.items():
    groups = set(zip(arrays["layer"].tolist(), arrays["channel"].tolist()))
    print(
        f"{part}: targets={len(arrays['bits'])}, "
        f"observed_layer_channel_groups={len(groups)}, "
        f"context_shape={arrays['context'].shape}"
    )

print("\nPASS: dataset validation completed using the existing validator.")
print("No predictor execution, test metrics, fitting, or file writes.")

train: targets=32256, observed_layer_channel_groups=5808, context_shape=(32256, 13)
calibration: targets=16034, observed_layer_channel_groups=2002, context_shape=(16034, 13)
test: targets=16061, observed_layer_channel_groups=2106, context_shape=(16061, 13)

PASS: dataset validation completed using the existing validator.
No predictor execution, test metrics, fitting, or file writes.


In [11]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
OUT = ROOT / "experiments/notebook06b/detection_repair_pilot_protocol_v1.json"
SESSION = ROOT / "results/notebook06a/masked_bit_comparison_v1/session.json"
session = json.loads(SESSION.read_bytes())["payload"]

protocol = {
    "version": "1.0",
    "status": "FROZEN_EXPLORATORY_PILOT_BEFORE_06B_CALIBRATION",
    "source_commit": "97aca930e3c9fdb47e714e9d49decf4778d894c0",
    "source_session_file_sha256": hashlib.sha256(SESSION.read_bytes()).hexdigest(),
    "dataset_sha256": session["dataset_sha256"],
    "representation": "05a_shared_logical_INT8_per_output_channel",
    "scope": "sampled_target_weights_in_63_Conv2d_layers",
    "methods": ["C", "D", "no_intervention"],
    "training": "reuse_verified_06a_models_without_refitting",
    "fault_model": {
        "type": "one_bit_flip_in_one_target_weight",
        "bits": list(range(8)),
        "each_scenario_starts_from_clean_state": True,
        "neighbors": "clean",
        "target_weight_under_inspection": "given",
        "injected_bit_position": "hidden_from_method"
    },
    "features": {
        "reuse": "06a_feature_definitions",
        "each_bit_predictor": "exclude_its_own_bit_from_observed_target",
        "other_bits": "observed_bits_including_fault_if_present",
        "context": "unchanged_target_excluded_same_output_channel",
        "forbidden": ["clean_target_value", "injected_bit", "address_as_feature"]
    },
    "decision": {
        "score": "p_if_observed_bit_is_0_else_1_minus_p",
        "candidate": "maximum_score_over_all_8_bits",
        "tie_break": "smallest_bit_index",
        "action": "flip_candidate_only_if_max_score_strictly_exceeds_threshold",
        "maximum_changes_per_target": 1
    },
    "calibration": {
        "partition": "calibration",
        "input": "clean_targets_only",
        "thresholds": "one_global_threshold_per_method_C_and_D",
        "engineering_false_intervention_budget": 0.01,
        "rule": (
            "For n clean targets, k=floor(0.01*n); "
            "threshold=sorted_max_scores[n-k-1] using zero_based_indexing; "
            "intervene_only_for_score>threshold."
        ),
        "ties": "conservative_no_randomization",
        "test_used_for_threshold_selection": False,
        "guarantee": "empirical_calibration_budget_only"
    },
    "evaluation": {
        "partition": "test",
        "clean_control": "one_clean_observation_per_target",
        "fault_cases": "8_separate_single_flip_scenarios_per_target",
        "primary": "exact_weight_recovery_fraction_over_all_fault_cases",
        "secondary": [
            "clean_false_intervention_fraction",
            "fault_intervention_fraction",
            "wrong_bit_intervention_fraction_over_all_fault_cases",
            "abstention_fraction_over_all_fault_cases",
            "correct_bit_fraction_among_fault_interventions",
            "per_bit_and_per_layer_results",
            "paired_C_minus_D_exact_recovery"
        ],
        "aggregation": "target_weighted_within_bit_then_equal_mean_over_8_bits",
        "uncertainty": {
            "unit": "output_channel",
            "stratify_by": "layer",
            "paired_methods_and_bits": True,
            "bootstrap_draws": 2000,
            "seed": 20261114,
            "interpretation": "exploratory_fixed_models_and_thresholds"
        },
        "test_evaluations": "one_after_thresholds_are_saved"
    },
    "limitations": [
        "06a test results were previously inspected; this is exploratory.",
        "Target under inspection is given; no network-wide localization claim.",
        "Clean neighbors; no multiple-fault robustness claim.",
        "No detection_AP_recovery claim from this weight-level pilot.",
        "One checkpoint; no cross-model generalization claim.",
        "Calibration false-intervention budget is not a test guarantee."
    ],
    "current_05b_campaign_changed": False
}

content = (
    json.dumps(protocol, indent=2, ensure_ascii=False, allow_nan=False) + "\n"
).encode("utf-8")

OUT.parent.mkdir(parents=True, exist_ok=True)
if OUT.exists():
    if OUT.read_bytes() != content:
        raise RuntimeError("Existing protocol differs; nothing overwritten.")
    print("IDENTICAL PROTOCOL ALREADY EXISTS")
else:
    with OUT.open("xb") as stream:
        stream.write(content)
    print("NEW PROTOCOL SAVED")

if OUT.read_bytes() != content:
    raise RuntimeError("Saved content verification failed.")

print("Path:", OUT)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No fitting, calibration, test evaluation, or changes to 05b.")

NEW PROTOCOL SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook06b/detection_repair_pilot_protocol_v1.json
SHA256: c996c54c5d8c14a0c57267617f4967ce48b2343a7c43e03669aff48d52ba1361
No fitting, calibration, test evaluation, or changes to 05b.


In [12]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
module_path = ROOT / "src/calibration06b_v1.py"

source = r'''"""Calibrate the frozen 06b pilot using saved clean calibration predictions."""
import hashlib
import io
import json
from pathlib import Path

PROTOCOL_SHA = "c996c54c5d8c14a0c57267617f4967ce48b2343a7c43e03669aff48d52ba1361"

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def digest(content):
    return hashlib.sha256(content).hexdigest()

def verified(path, expected):
    content = path.read_bytes()
    if digest(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def record(path):
    envelope = json.loads(path.read_bytes())
    if digest(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload mismatch: {path.name}")
    return envelope["payload"]

def run(repo):
    import numpy as np
    repo = Path(repo)
    protocol_path = repo / "experiments/notebook06b/detection_repair_pilot_protocol_v1.json"
    protocol = json.loads(verified(protocol_path, PROTOCOL_SHA))
    previous = repo / "results/notebook06a/masked_bit_comparison_v1"
    session_bytes = verified(
        previous / "session.json", protocol["source_session_file_sha256"]
    )
    session_envelope = json.loads(session_bytes)
    session = session_envelope["payload"]
    if digest(canonical(session)) != session_envelope["payload_sha256"]:
        raise RuntimeError("Session payload mismatch.")
    fingerprint = digest(canonical(session))

    dataset_path = repo / "results/notebook06a/masked_bit_dataset_v1/dataset.npz"
    raw = verified(dataset_path, protocol["dataset_sha256"])
    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
        bits = archive["calibration_bits"].copy()
    if bits.ndim != 2 or bits.shape[1] != 8 or not np.isin(bits, [0, 1]).all():
        raise RuntimeError("Invalid calibration labels.")

    n = len(bits)
    if n == 0:
        raise RuntimeError("Empty calibration partition.")
    scores = {method: np.empty((n, 8)) for method in ("C", "D")}
    prediction_hashes = {}
    for bit in range(8):
        saved = record(previous / f"bit_{bit}_result.json")
        if saved["bit"] != bit or saved["session_fingerprint"] != fingerprint:
            raise RuntimeError("Bit provenance mismatch.")
        prediction_path = previous / f"bit_{bit}_probabilities.npz"
        raw = verified(prediction_path, saved["prediction_sha256"])
        prediction_hashes[str(bit)] = saved["prediction_sha256"]
        with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
            for method in scores:
                p = archive[f"calibration_{method}"].copy()
                if (
                    p.shape != (n,) or not np.isfinite(p).all()
                    or np.any(p < 0) or np.any(p > 1)
                ):
                    raise RuntimeError("Invalid calibration probabilities.")
                scores[method][:, bit] = np.where(bits[:, bit] == 0, p, 1 - p)

    # Use exact integer arithmetic for the frozen one-percent budget.
    k = n // 100
    methods = {}
    for method, values in scores.items():
        maximum = values.max(axis=1)
        threshold = float(np.sort(maximum)[n - k - 1])
        count = int(np.count_nonzero(maximum > threshold))
        if count > k:
            raise RuntimeError("Calibration budget exceeded.")
        methods[method] = {
            "threshold": threshold,
            "comparison": "strictly_greater",
            "clean_targets": n,
            "allowed_interventions": k,
            "observed_interventions": count,
            "observed_fraction": count / n
        }

    payload = {
        "status": "CALIBRATION_COMPLETE_NO_TEST_EVALUATION",
        "protocol_sha256": PROTOCOL_SHA,
        "source_session_fingerprint": fingerprint,
        "dataset_sha256": protocol["dataset_sha256"],
        "prediction_sha256_by_bit": prediction_hashes,
        "module_sha256": digest(Path(__file__).read_bytes()),
        "numpy_version": np.__version__,
        "methods": methods,
        "limitation": "Empirical clean calibration budget; no test-rate guarantee."
    }
    envelope = {"payload_sha256": digest(canonical(payload)), "payload": payload}
    content = (json.dumps(envelope, indent=2, allow_nan=False) + "\n").encode()
    out = repo / "results/notebook06b/calibration_v1.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists():
        if out.read_bytes() != content:
            raise RuntimeError("Existing calibration differs; nothing overwritten.")
    else:
        with out.open("xb") as stream:
            stream.write(content)
    if out.read_bytes() != content:
        raise RuntimeError("Write verification failed.")
    print(json.dumps(methods, indent=2))
    print("Saved:", out)
    print("SHA256:", digest(content))
    print("No test arrays or test probabilities accessed; no model fitting.")
    return payload
'''

# Validate syntax before creating the module.
compile(source, str(module_path), "exec")
content = source.encode("utf-8")

if module_path.exists():
    if module_path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
    print("IDENTICAL MODULE ALREADY EXISTS")
else:
    with module_path.open("xb") as stream:
        stream.write(content)
    print("NEW MODULE SAVED")

assert module_path.read_bytes() == content
print("Path:", module_path)
print("Syntax: PASS")
print("Calibration has not been executed.")

NEW MODULE SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/src/calibration06b_v1.py
Syntax: PASS
Calibration has not been executed.


In [13]:
import importlib.util
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/calibration06b_v1.py"

spec = importlib.util.spec_from_file_location("calibration06b_v1", path)
calibration06b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(calibration06b)

calibration_result = calibration06b.run(ROOT)

{
  "C": {
    "threshold": 0.9764253660024884,
    "comparison": "strictly_greater",
    "clean_targets": 16034,
    "allowed_interventions": 160,
    "observed_interventions": 160,
    "observed_fraction": 0.009978795060496445
  },
  "D": {
    "threshold": 0.9757962345778055,
    "comparison": "strictly_greater",
    "clean_targets": 16034,
    "allowed_interventions": 160,
    "observed_interventions": 160,
    "observed_fraction": 0.009978795060496445
  }
}
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook06b/calibration_v1.json
SHA256: 87bfe4072cc4f8b1eb90fe961d4c348552f674223d6d49a85f5e7e0c7e1931e0
No test arrays or test probabilities accessed; no model fitting.


In [14]:
from pathlib import Path
import importlib.util
import io
import json
import hashlib
import joblib
import numpy as np
from threadpoolctl import threadpool_limits

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
PREVIOUS = ROOT / "results/notebook06a/masked_bit_comparison_v1"

spec = importlib.util.spec_from_file_location(
    "contextual06a_reuse", ROOT / "src/contextual06a_runner_v1.py"
)
runner06a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner06a)

session = runner06a.read_json(PREVIOUS / "session.json")
fingerprint = runner06a.sha256(runner06a.canonical(session))

raw = runner06a.verified(
    ROOT / "results/notebook06a/masked_bit_dataset_v1/dataset.npz",
    session["dataset_sha256"],
)
with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
    sample = {
        key: archive[f"calibration_{key}"][:32].copy()
        for key in ("bits", "context", "layer", "channel", "flat_index")
    }

with threadpool_limits(limits=1):
    for bit in range(8):
        record = runner06a.read_json(PREVIOUS / f"bit_{bit}_result.json")
        if record["bit"] != bit or record["session_fingerprint"] != fingerprint:
            raise RuntimeError("Bit provenance mismatch.")

        model_bytes = runner06a.verified(
            PREVIOUS / f"bit_{bit}_models.joblib", record["models_sha256"]
        )
        prediction_bytes = runner06a.verified(
            PREVIOUS / f"bit_{bit}_probabilities.npz",
            record["prediction_sha256"],
        )

        # Load only the previously verified, locally produced model artifact.
        models = joblib.load(io.BytesIO(model_bytes))
        predicted = runner06a.predict_models(models, sample, bit)

        with np.load(io.BytesIO(prediction_bytes), allow_pickle=False) as archive:
            for method in ("C", "D"):
                expected = archive[f"calibration_{method}"][:32]
                actual = predicted[method]
                np.testing.assert_allclose(
                    actual, expected, rtol=0, atol=1e-12
                )
                print(
                    f"PASS: bit={bit}, method={method}, "
                    f"max_abs_difference={np.max(np.abs(actual - expected)):.3g}"
                )

print("\nPASS: saved models reproduce the checked calibration predictions.")
print("Only 32 calibration targets checked; no test evaluation or file writes.")

PASS: bit=0, method=C, max_abs_difference=0
PASS: bit=0, method=D, max_abs_difference=0
PASS: bit=1, method=C, max_abs_difference=0
PASS: bit=1, method=D, max_abs_difference=0
PASS: bit=2, method=C, max_abs_difference=0
PASS: bit=2, method=D, max_abs_difference=0
PASS: bit=3, method=C, max_abs_difference=0
PASS: bit=3, method=D, max_abs_difference=0
PASS: bit=4, method=C, max_abs_difference=0
PASS: bit=4, method=D, max_abs_difference=0
PASS: bit=5, method=C, max_abs_difference=0
PASS: bit=5, method=D, max_abs_difference=0
PASS: bit=6, method=C, max_abs_difference=0
PASS: bit=6, method=D, max_abs_difference=0
PASS: bit=7, method=C, max_abs_difference=0
PASS: bit=7, method=D, max_abs_difference=0

PASS: saved models reproduce the checked calibration predictions.
Only 32 calibration targets checked; no test evaluation or file writes.


In [15]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/evaluation06b_v1.py"

source = r'''"""Frozen exploratory weight-level evaluation; no fitting or threshold tuning."""
import hashlib
import importlib.util
import io
import json
import platform
import time
from pathlib import Path

CALIBRATION_SHA = "87bfe4072cc4f8b1eb90fe961d4c348552f674223d6d49a85f5e7e0c7e1931e0"
PROTOCOL_SHA = "c996c54c5d8c14a0c57267617f4967ce48b2343a7c43e03669aff48d52ba1361"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def unpack(content):
    envelope = json.loads(content)
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Record payload checksum mismatch.")
    return envelope["payload"]

def run(repo):
    import joblib
    import numpy as np
    from importlib.metadata import version
    from threadpoolctl import threadpool_limits

    repo = Path(repo)
    out = repo / "results/notebook06b/evaluation_v1"
    if out.exists():
        raise FileExistsError("Evaluation directory exists; inspect before any rerun.")

    protocol = json.loads(verified(
        repo / "experiments/notebook06b/detection_repair_pilot_protocol_v1.json",
        PROTOCOL_SHA
    ))
    calibration = unpack(verified(
        repo / "results/notebook06b/calibration_v1.json", CALIBRATION_SHA
    ))
    if calibration["protocol_sha256"] != PROTOCOL_SHA:
        raise RuntimeError("Calibration protocol mismatch.")

    previous = repo / "results/notebook06a/masked_bit_comparison_v1"
    session = unpack(verified(
        previous / "session.json", protocol["source_session_file_sha256"]
    ))
    fingerprint = sha(canonical(session))
    if calibration["source_session_fingerprint"] != fingerprint:
        raise RuntimeError("Calibration session mismatch.")
    for package, expected in session["environment"]["packages"].items():
        if version(package) != expected:
            raise RuntimeError(f"Package version mismatch: {package}")

    runner_path = repo / "src/contextual06a_runner_v1.py"
    verified(runner_path, session["source_sha256"])
    spec = importlib.util.spec_from_file_location("trusted06a_eval", runner_path)
    runner = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(runner)

    raw = verified(
        repo / "results/notebook06a/masked_bit_dataset_v1/dataset.npz",
        protocol["dataset_sha256"]
    )
    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
        test = {
            key: archive[f"test_{key}"].copy()
            for key in ("bits", "context", "layer", "channel", "flat_index")
        }
    n = len(test["bits"])
    if n == 0 or test["bits"].shape != (n, 8):
        raise RuntimeError("Invalid test shape.")
    if not np.isin(test["bits"], [0, 1]).all():
        raise RuntimeError("Invalid test bits.")

    methods = ("C", "D")
    models = {}
    clean_scores = {m: np.empty((n, 8)) for m in methods}
    thresholds = {m: calibration["methods"][m]["threshold"] for m in methods}
    started = time.perf_counter()

    with threadpool_limits(limits=1):
        for bit in range(8):
            record = unpack((previous / f"bit_{bit}_result.json").read_bytes())
            if record["bit"] != bit or record["session_fingerprint"] != fingerprint:
                raise RuntimeError("Bit provenance mismatch.")
            if record["prediction_sha256"] != calibration["prediction_sha256_by_bit"][str(bit)]:
                raise RuntimeError("Prediction provenance mismatch.")
            model_bytes = verified(
                previous / f"bit_{bit}_models.joblib", record["models_sha256"]
            )
            models[bit] = joblib.load(io.BytesIO(model_bytes))
            raw_predictions = verified(
                previous / f"bit_{bit}_probabilities.npz",
                record["prediction_sha256"]
            )
            with np.load(io.BytesIO(raw_predictions), allow_pickle=False) as archive:
                for m in methods:
                    p = archive[f"test_{m}"].copy()
                    if p.shape != (n,) or not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
                        raise RuntimeError("Invalid saved probabilities.")
                    clean_scores[m][:, bit] = np.where(test["bits"][:, bit] == 0, p, 1-p)

        arrays = {
            key: test[key] for key in ("layer", "channel", "flat_index")
        }
        summary = {}
        for m in methods:
            arrays[f"{m}_clean_intervention"] = clean_scores[m].max(axis=1) > thresholds[m]
            arrays[f"{m}_fault_candidate"] = np.empty((n, 8), dtype=np.uint8)
            arrays[f"{m}_fault_intervention"] = np.empty((n, 8), dtype=bool)

        for injected_bit in range(8):
            observed = dict(test)
            observed["bits"] = test["bits"].copy()
            observed["bits"][:, injected_bit] ^= 1
            scores = {m: np.empty((n, 8)) for m in methods}

            for candidate_bit in range(8):
                for m in methods:
                    estimator = models[candidate_bit][m]
                    features = runner.features(
                        observed, candidate_bit, with_context=(m == "C")
                    )
                    if len(estimator.classes_) == 1:
                        p = np.full(n, float(estimator.classes_[0]))
                    else:
                        probabilities = estimator.predict_proba(features)
                        ones = np.flatnonzero(estimator.classes_ == 1)
                        p = probabilities[:, int(ones[0])] if len(ones) else np.zeros(n)
                    if not np.isfinite(p).all() or np.any((p < 0) | (p > 1)):
                        raise RuntimeError("Invalid fault-case probabilities.")
                    scores[m][:, candidate_bit] = np.where(
                        observed["bits"][:, candidate_bit] == 0, p, 1-p
                    )

            for m in methods:
                # NumPy argmax selects the first index when scores tie.
                arrays[f"{m}_fault_candidate"][:, injected_bit] = scores[m].argmax(axis=1)
                arrays[f"{m}_fault_intervention"][:, injected_bit] = (
                    scores[m].max(axis=1) > thresholds[m]
                )
            print(f"Completed injected bit {injected_bit}/7", flush=True)

        for m in methods:
            acted = arrays[f"{m}_fault_intervention"]
            correct = arrays[f"{m}_fault_candidate"] == np.arange(8)[None, :]
            recovered = acted & correct
            wrong = acted & ~correct
            arrays[f"{m}_exact_recovery"] = recovered
            summary[m] = {
                "threshold": thresholds[m],
                "clean_false_intervention_fraction": float(arrays[f"{m}_clean_intervention"].mean()),
                "fault_intervention_fraction": float(acted.mean()),
                "exact_recovery_fraction": float(recovered.mean()),
                "wrong_bit_intervention_fraction": float(wrong.mean()),
                "abstention_fraction": float((~acted).mean()),
                "correct_bit_fraction_among_interventions": (
                    float(recovered.sum() / acted.sum()) if acted.any() else None
                ),
                "exact_recovery_by_injected_bit": recovered.mean(axis=0).tolist()
            }
        summary["no_intervention"] = {
            "clean_false_intervention_fraction": 0.0,
            "fault_intervention_fraction": 0.0,
            "exact_recovery_fraction": 0.0,
            "wrong_bit_intervention_fraction": 0.0,
            "abstention_fraction": 1.0
        }

    buffer = io.BytesIO()
    np.savez_compressed(buffer, **arrays)
    outcome_bytes = buffer.getvalue()
    payload = {
        "status": "EXPLORATORY_WEIGHT_LEVEL_TEST_COMPLETE",
        "protocol_sha256": PROTOCOL_SHA,
        "calibration_sha256": CALIBRATION_SHA,
        "source_session_fingerprint": fingerprint,
        "module_sha256": sha(Path(__file__).read_bytes()),
        "clean_targets": n,
        "fault_scenarios_per_method": n * 8,
        "methods": summary,
        "outcomes_sha256": sha(outcome_bytes),
        "python": platform.python_version(),
        "packages": {p: version(p) for p in session["environment"]["packages"]},
        "wall_seconds": time.perf_counter() - started,
        "limitations": protocol["limitations"],
        "uncertainty": "Pending analysis of saved outcomes; no repeated model evaluation needed."
    }
    envelope = {"payload_sha256": sha(canonical(payload)), "payload": payload}
    summary_bytes = (json.dumps(envelope, indent=2, allow_nan=False) + "\n").encode()

    # Preserve a single evaluation attempt; never overwrite existing outputs.
    out.mkdir(parents=True, exist_ok=False)
    for name, content in (
        ("outcomes.npz", outcome_bytes), ("summary.json", summary_bytes)
    ):
        with (out / name).open("xb") as stream:
            stream.write(content)
        if (out / name).read_bytes() != content:
            raise RuntimeError("Saved output verification failed.")

    print(json.dumps(summary, indent=2))
    print("Saved:", out)
    print("No fitting, threshold adjustment, COCO inference, or changes to 05b.")
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
    print("IDENTICAL MODULE ALREADY EXISTS")
else:
    with path.open("xb") as stream:
        stream.write(content)
    print("NEW MODULE SAVED")

assert path.read_bytes() == content
print("Path:", path)
print("Syntax: PASS")
print("Test evaluation has not been executed.")

NEW MODULE SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/src/evaluation06b_v1.py
Syntax: PASS
Test evaluation has not been executed.


In [16]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/evaluation06b_v1.py"

spec = importlib.util.spec_from_file_location("evaluation06b_v1", path)
evaluation06b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(evaluation06b)

evaluation_result = evaluation06b.run(ROOT)

Completed injected bit 0/7
Completed injected bit 1/7
Completed injected bit 2/7
Completed injected bit 3/7
Completed injected bit 4/7
Completed injected bit 5/7
Completed injected bit 6/7
Completed injected bit 7/7
{
  "C": {
    "threshold": 0.9764253660024884,
    "clean_false_intervention_fraction": 0.009588444057032562,
    "fault_intervention_fraction": 0.19468744162879023,
    "exact_recovery_fraction": 0.1742419525558807,
    "wrong_bit_intervention_fraction": 0.020445489072909533,
    "abstention_fraction": 0.8053125583712097,
    "correct_bit_fraction_among_interventions": 0.8949830101938837,
    "exact_recovery_by_injected_bit": [
      0.0,
      0.0,
      0.0,
      0.0021169292073968,
      0.044953614345308514,
      0.15490940788244817,
      0.4860220409688064,
      0.7059336280430857
    ]
  },
  "D": {
    "threshold": 0.9757962345778055,
    "clean_false_intervention_fraction": 0.0076583027208766575,
    "fault_intervention_fraction": 0.142410186165245,
    "exact

In [17]:
from pathlib import Path
import importlib.util
import io
import numpy as np

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
OUT = ROOT / "results/notebook06b/evaluation_v1"

spec = importlib.util.spec_from_file_location(
    "contextual06a_analysis", ROOT / "src/contextual06a_runner_v1.py"
)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

summary = runner.read_json(OUT / "summary.json")
raw = runner.verified(OUT / "outcomes.npz", summary["outcomes_sha256"])

with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
    arrays = {name: archive[name].copy() for name in archive.files}

n = summary["clean_targets"]
for method in ("C", "D"):
    acted = arrays[f"{method}_fault_intervention"]
    candidate = arrays[f"{method}_fault_candidate"]
    recovered = arrays[f"{method}_exact_recovery"]
    clean = arrays[f"{method}_clean_intervention"]

    assert acted.shape == candidate.shape == recovered.shape == (n, 8)
    assert clean.shape == (n,)
    assert acted.dtype == recovered.dtype == clean.dtype == np.dtype(bool)
    assert np.all(candidate < 8)
    assert np.array_equal(
        recovered, acted & (candidate == np.arange(8)[None, :])
    )
    assert float(recovered.mean()) == summary["methods"][method]["exact_recovery_fraction"]
    assert float(clean.mean()) == summary["methods"][method]["clean_false_intervention_fraction"]

recovery_delta = (
    arrays["C_exact_recovery"].astype(float)
    - arrays["D_exact_recovery"].astype(float)
)
clean_delta = (
    arrays["C_clean_intervention"].astype(float)
    - arrays["D_clean_intervention"].astype(float)
)

# Use the protocol's frozen bootstrap seed and number of draws.
runner.BOOTSTRAP_SEED = 20261114

recovery = runner.paired_channel_bootstrap(
    recovery_delta, arrays["layer"], arrays["channel"], draws=2000
)
clean = runner.paired_channel_bootstrap(
    clean_delta, arrays["layer"], arrays["channel"], draws=2000
)

analysis = {
    "status": "EXPLORATORY_PAIRED_CHANNEL_BOOTSTRAP",
    "evaluation_summary_payload_sha256": runner.sha256(runner.canonical(summary)),
    "outcomes_sha256": summary["outcomes_sha256"],
    "bootstrap_source_sha256": runner.sha256(
        (ROOT / "src/contextual06a_runner_v1.py").read_bytes()
    ),
    "C_minus_D_exact_recovery": recovery,
    "C_minus_D_clean_false_intervention": clean,
    "interpretation": {
        "recovery": "Positive difference favors C.",
        "clean_false_intervention": "Positive difference means more harm to clean weights with C.",
        "scope": "Exploratory; fixed models and thresholds; no AP recovery claim."
    }
}

destination = OUT / "paired_uncertainty_v1.json"
runner.write_json(destination, analysis)

for label, result in (
    ("Exact recovery: C minus D", recovery),
    ("Clean false intervention: C minus D", clean),
):
    print(label)
    print("Difference (percentage points):", 100 * result["mean_over_bits"])
    print(
        "Exploratory 95% interval (percentage points):",
        [
            100 * value
            for value in result[
                "exploratory_percentile_95_interval_mean_over_bits"
            ]
        ],
    )
    print("Observed channel clusters:", result["test_output_channel_clusters"])

print("Saved:", destination)
print("No model evaluation, fitting, or threshold changes.")

Exact recovery: C minus D
Difference (percentage points): 5.166241205404395
Exploratory 95% interval (percentage points): [4.8775724342183056, 5.453157722650575]
Observed channel clusters: 2106
Clean false intervention: C minus D
Difference (percentage points): 0.19301413361559056
Exploratory 95% interval (percentage points): [0.030743483852850865, 0.3552890824818075]
Observed channel clusters: 2106
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook06b/evaluation_v1/paired_uncertainty_v1.json
No model evaluation, fitting, or threshold changes.


In [18]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
SOURCE = ROOT / "results/notebook06b/evaluation_v1"

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def read_verified_record(path):
    envelope = json.loads(path.read_bytes())
    payload = envelope["payload"]
    if hashlib.sha256(canonical(payload)).hexdigest() != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload mismatch: {path.name}")
    return payload

summary = read_verified_record(SOURCE / "summary.json")
analysis = read_verified_record(SOURCE / "paired_uncertainty_v1.json")

if analysis["evaluation_summary_payload_sha256"] != hashlib.sha256(
    canonical(summary)
).hexdigest():
    raise RuntimeError("Analysis does not match evaluation summary.")

if analysis["outcomes_sha256"] != summary["outcomes_sha256"]:
    raise RuntimeError("Outcome references differ.")

lines = [
    "# Notebook 06b: exploratory detection and repair pilot",
    "",
    "## Scope",
    f"- Clean test targets: {summary['clean_targets']}.",
    f"- Single-bit fault scenarios per method: {summary['fault_scenarios_per_method']}.",
    "- Fixed checkpoint, saved C/D predictors, frozen calibration thresholds.",
    "- The target weight under inspection is given; the injected bit is hidden.",
    "- Neighbors are clean; at most one bit is changed by the method.",
    "",
    "## Test results",
    "",
    "| Metric | C | D |",
    "| --- | ---: | ---: |",
]

for label, key in [
    ("Exact weight recovery (%)", "exact_recovery_fraction"),
    ("Clean false intervention (%)", "clean_false_intervention_fraction"),
    ("Wrong-bit intervention / all fault cases (%)", "wrong_bit_intervention_fraction"),
    ("Abstention / all fault cases (%)", "abstention_fraction"),
]:
    c = 100 * summary["methods"]["C"][key]
    d = 100 * summary["methods"]["D"][key]
    lines.append(f"| {label} | {c:.6f} | {d:.6f} |")

lines += [
    "",
    "## Exact recovery by injected bit",
    "",
    "| Bit | C (%) | D (%) |",
    "| ---: | ---: | ---: |",
]
for bit in range(8):
    c = 100 * summary["methods"]["C"]["exact_recovery_by_injected_bit"][bit]
    d = 100 * summary["methods"]["D"]["exact_recovery_by_injected_bit"][bit]
    lines.append(f"| {bit} | {c:.6f} | {d:.6f} |")

lines += ["", "## Paired exploratory uncertainty", ""]
for label, key in [
    ("Exact recovery", "C_minus_D_exact_recovery"),
    ("Clean false intervention", "C_minus_D_clean_false_intervention"),
]:
    result = analysis[key]
    difference = 100 * result["mean_over_bits"]
    low, high = [
        100 * value for value in
        result["exploratory_percentile_95_interval_mean_over_bits"]
    ]
    lines.append(
        f"- {label}, C minus D: {difference:.6f} percentage points; "
        f"exploratory 95% interval [{low:.6f}, {high:.6f}]."
    )

lines += [
    "",
    "## Interpretation and limits",
    "",
    "- C improves average exact recovery relative to D but increases clean false interventions.",
    "- Both methods have zero exact recovery for injected bits 0, 1 and 2.",
    "- D has higher exact recovery for injected bit 7.",
    "- The 1% budget was enforced on calibration samples; it is not a test guarantee.",
    "- Bootstrap resamples observed output channels within layers with paired methods and bits.",
    "- Intervals condition on fixed models and thresholds; they are exploratory.",
    "- Prior 06a test results were inspected; this is not independent confirmation.",
    "- No network-wide localization, deployment benefit, novelty, or AP recovery is established.",
    "- Per-layer outcome analysis and end-to-end evaluation remain pending.",
    "",
    "## Provenance",
    "",
    f"- Protocol SHA256: {summary['protocol_sha256']}",
    f"- Calibration SHA256: {summary['calibration_sha256']}",
    f"- Outcome archive SHA256: {summary['outcomes_sha256']}",
    "- Detailed records: results/notebook06b/evaluation_v1/",
    "- No changes to the 05b campaign.",
    "",
]

destination = ROOT / "docs/research_reports/notebook06b_pilot_v1.md"
content = "\n".join(lines).encode("utf-8")
destination.parent.mkdir(parents=True, exist_ok=True)

if destination.exists():
    if destination.read_bytes() != content:
        raise RuntimeError("Existing report differs; nothing overwritten.")
    print("IDENTICAL REPORT ALREADY EXISTS")
else:
    with destination.open("xb") as stream:
        stream.write(content)
    print("NEW REPORT SAVED")

assert destination.read_bytes() == content
print("Path:", destination)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No experiments rerun; no commit or push.")

NEW REPORT SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/notebook06b_pilot_v1.md
SHA256: 5e4632276a64de4679ea5383761795ca31c79365bba354f657de7cee37e8e7d9
No experiments rerun; no commit or push.


In [19]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/layer_analysis06b_v1.py"

source = r'''"""Descriptive per-layer analysis of saved 06b outcomes."""
import csv
import hashlib
import io
import json
from pathlib import Path

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode()

def read_record(path):
    envelope = json.loads(path.read_bytes())
    if hashlib.sha256(canonical(envelope["payload"])).hexdigest() != envelope["payload_sha256"]:
        raise RuntimeError("Record checksum mismatch.")
    return envelope["payload"]

def run(repo):
    import numpy as np
    repo = Path(repo)
    previous = repo / "results/notebook06b/evaluation_v1"
    summary = read_record(previous / "summary.json")
    raw = (previous / "outcomes.npz").read_bytes()
    if hashlib.sha256(raw).hexdigest() != summary["outcomes_sha256"]:
        raise RuntimeError("Outcome checksum mismatch.")

    manifest_path = repo / "results/notebook06a/masked_bit_dataset_v1/manifest.json"
    session = read_record(
        repo / "results/notebook06a/masked_bit_comparison_v1/session.json"
    )
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != session["manifest_sha256"]:
        raise RuntimeError("Manifest checksum mismatch.")
    if hashlib.sha256(canonical(session)).hexdigest() != summary["source_session_fingerprint"]:
        raise RuntimeError("Source session mismatch.")
    manifest = json.loads(manifest_bytes)

    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
        arrays = {key: archive[key].copy() for key in archive.files}

    rows = []
    for item in manifest["inventory"]:
        mask = arrays["layer"] == item["layer_id"]
        n = int(mask.sum())
        if n == 0:
            raise RuntimeError("Missing test layer.")
        row = {
            "layer_id": item["layer_id"],
            "layer": item["layer"],
            "test_targets": n,
            "observed_test_channels": int(len(np.unique(arrays["channel"][mask]))),
            "fault_scenarios_per_method": n * 8,
        }
        for method in ("C", "D"):
            acted = arrays[f"{method}_fault_intervention"][mask]
            recovered = arrays[f"{method}_exact_recovery"][mask]
            row[f"{method}_recovery_pct"] = float(recovered.mean() * 100)
            row[f"{method}_clean_false_intervention_pct"] = float(
                arrays[f"{method}_clean_intervention"][mask].mean() * 100
            )
            row[f"{method}_wrong_bit_intervention_pct"] = float(
                (acted & ~recovered).mean() * 100
            )
            for bit in range(8):
                row[f"{method}_recovery_bit_{bit}_pct"] = float(
                    recovered[:, bit].mean() * 100
                )
        row["recovery_C_minus_D_pp"] = row["C_recovery_pct"] - row["D_recovery_pct"]
        row["clean_harm_C_minus_D_pp"] = (
            row["C_clean_false_intervention_pct"]
            - row["D_clean_false_intervention_pct"]
        )
        rows.append(row)

    if len(rows) != 63 or sum(r["test_targets"] for r in rows) != summary["clean_targets"]:
        raise RuntimeError("Layer coverage mismatch.")

    stream = io.StringIO(newline="")
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
    content = stream.getvalue().encode("utf-8")
    out = repo / "docs/research_reports/notebook06b_per_layer_v1.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists():
        if out.read_bytes() != content:
            raise RuntimeError("Existing table differs; nothing overwritten.")
    else:
        with out.open("xb") as handle:
            handle.write(content)
    if out.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

    better = sum(r["recovery_C_minus_D_pp"] > 0 for r in rows)
    equal = sum(r["recovery_C_minus_D_pp"] == 0 for r in rows)
    worse = len(rows) - better - equal
    print(f"Layers: {len(rows)} | C recovery higher: {better}, equal: {equal}, lower: {worse}")
    print("Layer | targets | C recovery % | D recovery % | recovery delta pp | clean harm delta pp")
    for row in rows:
        print(
            f"{row['layer']} | {row['test_targets']} | "
            f"{row['C_recovery_pct']:.3f} | {row['D_recovery_pct']:.3f} | "
            f"{row['recovery_C_minus_D_pp']:+.3f} | "
            f"{row['clean_harm_C_minus_D_pp']:+.3f}"
        )
    print("Saved:", out)
    print("CSV SHA256:", hashlib.sha256(content).hexdigest())
    print("Descriptive only; no per-layer uncertainty or 05b sensitivity ranking.")
    return rows
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
else:
    with path.open("xb") as stream:
        stream.write(content)

assert path.read_bytes() == content
spec = importlib.util.spec_from_file_location("layer_analysis06b_v1", path)
layer_analysis06b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(layer_analysis06b)
layer_rows = layer_analysis06b.run(ROOT)

Layers: 63 | C recovery higher: 58, equal: 1, lower: 4
Layer | targets | C recovery % | D recovery % | recovery delta pp | clean harm delta pp
model.0.conv | 189 | 0.265 | 0.000 | +0.265 | +1.058
model.1.conv | 256 | 20.020 | 10.645 | +9.375 | +0.000
model.2.cv1.conv | 256 | 9.961 | 4.053 | +5.908 | +1.172
model.2.cv2.conv | 256 | 8.789 | 3.613 | +5.176 | -0.391
model.2.m.0.cv1.conv | 256 | 18.652 | 17.871 | +0.781 | +0.391
model.2.m.0.cv2.conv | 256 | 18.311 | 12.402 | +5.908 | +0.781
model.3.conv | 256 | 17.822 | 10.352 | +7.471 | -0.781
model.4.cv1.conv | 256 | 8.154 | 7.178 | +0.977 | +1.562
model.4.cv2.conv | 256 | 9.668 | 9.131 | +0.537 | -0.781
model.4.m.0.cv1.conv | 256 | 16.553 | 15.723 | +0.830 | +1.172
model.4.m.0.cv2.conv | 256 | 14.941 | 10.840 | +4.102 | +0.781
model.4.m.1.cv1.conv | 256 | 14.404 | 11.084 | +3.320 | +0.781
model.4.m.1.cv2.conv | 256 | 14.746 | 10.889 | +3.857 | +0.391
model.5.conv | 256 | 19.482 | 13.574 | +5.908 | -0.781
model.6.cv1.conv | 256 | 9.277 | 

In [20]:
from pathlib import Path
import ast

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

for relative in (
    "src/baseline05a_runner_v1.py",
    "src/fault05b_validation_v1.py",
    "src/screen05b_campaign_v1.py",
):
    path = ROOT / relative
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    print(f"\nFILE: {relative}")
    print(f"Total lines: {len(source.splitlines())}")

    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            print(ast.get_source_segment(source, node))
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            print(f"\nFunction: {node.name} | lines {node.lineno}-{node.end_lineno}")
            print("Arguments:", ast.unparse(node.args))
            docstring = ast.get_docstring(node)
            if docstring:
                print("Docstring:", docstring)

print("\nRead-only inspection; no project modules imported or executed.")


FILE: src/baseline05a_runner_v1.py
Total lines: 402
from collections import Counter
from datetime import datetime, timezone
import copy
import gzip
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import time

Function: canonical | lines 27-29
Arguments: value

Function: digest | lines 32-33
Arguments: data

Function: read_verified | lines 36-40
Arguments: path, expected

Function: validate_split | lines 43-52
Arguments: split

Function: write_record | lines 55-74
Arguments: path, payload
Docstring: Verified exclusive publication with a crash-visible partial file.

Function: read_record | lines 77-83
Arguments: path

Function: validate_chunk | lines 86-112
Arguments: payload, fingerprint, image_ids

Function: prepare | lines 401-402
Arguments: project_root

FILE: src/fault05b_validation_v1.py
Total lines: 290
import hashlib
import json
import time
from pathlib import Path

Function: flip_int8 | lines 19-25
Arguments: q, bit

Function: verify_su

In [21]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

for relative in (
    "src/baseline05a_runner_v1.py",
    "src/fault05b_validation_v1.py",
    "src/screen05b_campaign_v1.py",
):
    path = ROOT / relative
    lines = path.read_text(encoding="utf-8").splitlines()

    print(f"\n{'=' * 60}")
    print(f"FILE: {relative} | Total lines: {len(lines)}")
    print("=" * 60)

    for number, line in enumerate(lines, start=1):
        print(f"{number:04d}: {line}")


FILE: src/baseline05a_runner_v1.py | Total lines: 402
0001: """CPU-only paired FP32/W8A32 clean validation; no fault injection.
0002: 
0003: Use one writer per output directory. Existing results are never overwritten.
0004: GPU support, fault screening and hardware INT8 claims are outside this module.
0005: """
0006: 
0007: from collections import Counter
0008: from datetime import datetime, timezone
0009: import copy
0010: import gzip
0011: import hashlib
0012: import json
0013: import math
0014: import os
0015: from pathlib import Path
0016: import platform
0017: import time
0018: 
0019: VERSION = "05a-clean-cpu-v1"
0020: CHECKPOINT_SHA = "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
0021: SPLIT_SHA = "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
0022: GATE_SHA = "dfc04556e731df68af9375c161778e41b87f66601856265703ea8acd69fb64f3"
0023: MODELS = ("fp32_clean", "int8_channel_clean")
0024: PACKAGES = ("torch", "torchvision", "ultralytics", "nump

In [22]:
from pathlib import Path
import hashlib
import json
import numpy as np

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
plan_path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
plan_bytes = plan_path.read_bytes()

expected = json.loads(
    plan_path.with_suffix(".sha256.json").read_bytes()
)["sha256"]
if hashlib.sha256(plan_bytes).hexdigest() != expected:
    raise RuntimeError("Screening plan checksum mismatch.")

plan = json.loads(plan_bytes)
matches = [
    row for row in plan["scenarios"]
    if row["scenario_id"] == "S05B-00731"
]
if len(matches) != 1:
    raise RuntimeError("Expected exactly one matching scenario.")
scenario = matches[0]

inventory = next(
    row for row in plan["inventory"] if row["name"] == scenario["layer"]
)
shape = inventory["weight_shape"]
weights_per_channel = int(np.prod(shape[1:]))
channel = scenario["flat_index"] // weights_per_channel

manifest_path = ROOT / "results/notebook06a/masked_bit_dataset_v1/manifest.json"
manifest_bytes = manifest_path.read_bytes()
if hashlib.sha256(manifest_bytes).hexdigest() != (
    "515632610a2503bce69fd99b21391d37968d217457c54f39d56f73143ea307fe"
):
    raise RuntimeError("06a manifest checksum mismatch.")

manifest = json.loads(manifest_bytes)
layer = next(
    row for row in manifest["inventory"]
    if row["layer"] == scenario["layer"]
)

print("Scenario:")
print(json.dumps(scenario, indent=2))
print("Weight shape:", shape)
print("Output channel:", channel)

dataset_path = ROOT / "results/notebook06a/masked_bit_dataset_v1/dataset.npz"
dataset_bytes = dataset_path.read_bytes()
if hashlib.sha256(dataset_bytes).hexdigest() != manifest["dataset_sha256"]:
    raise RuntimeError("06a dataset checksum mismatch.")

with np.load(__import__("io").BytesIO(dataset_bytes), allow_pickle=False) as archive:
    for part in ("train", "calibration", "test"):
        sampled = (
            (archive[f"{part}_layer"] == layer["layer_id"])
            & (archive[f"{part}_flat_index"] == scenario["flat_index"])
        )
        print(
            f"{part}: channel_assigned={channel in layer['channels'][part]}, "
            f"exact_target_sampled={bool(sampled.any())}"
        )

print("No model loading, inference, or file changes.")

Scenario:
{
  "layer": "model.0.conv",
  "bit": 7,
  "replicate": 1,
  "flat_index": 475,
  "scenario_id": "S05B-00731",
  "q_clean": -69,
  "q_faulty": 59,
  "scale": 0.002639410085976124,
  "clean_weight": -0.18211929500102997,
  "faulty_weight": 0.15572519600391388
}
Weight shape: [32, 3, 3, 3]
Output channel: 17
train: channel_assigned=True, exact_target_sampled=True
calibration: channel_assigned=False, exact_target_sampled=False
test: channel_assigned=False, exact_target_sampled=False
No model loading, inference, or file changes.


In [23]:
from pathlib import Path
import hashlib
import io
import json
import numpy as np

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
DEST = ROOT / "experiments/notebook06b/end_to_end_pilot_protocol_v1.json"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def read_record(path):
    envelope = json.loads(path.read_bytes())
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Record checksum mismatch.")
    return envelope["payload"]

session = read_record(
    ROOT / "results/notebook06a/masked_bit_comparison_v1/session.json"
)
source = ROOT / "results/notebook06a/masked_bit_dataset_v1"

manifest_bytes = (source / "manifest.json").read_bytes()
dataset_bytes = (source / "dataset.npz").read_bytes()
if sha(manifest_bytes) != session["manifest_sha256"]:
    raise RuntimeError("Manifest checksum mismatch.")
if sha(dataset_bytes) != session["dataset_sha256"]:
    raise RuntimeError("Dataset checksum mismatch.")
manifest = json.loads(manifest_bytes)

with np.load(io.BytesIO(dataset_bytes), allow_pickle=False) as archive:
    test = {
        key: archive[f"test_{key}"].copy()
        for key in ("bits", "layer", "channel", "flat_index")
    }

plan_path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
plan_bytes = plan_path.read_bytes()
if sha(plan_bytes) != json.loads(
    plan_path.with_suffix(".sha256.json").read_bytes()
)["sha256"]:
    raise RuntimeError("05b plan checksum mismatch.")
plan05b = json.loads(plan_bytes)

calibration_path = ROOT / "results/notebook06b/calibration_v1.json"
calibration_bytes = calibration_path.read_bytes()
if sha(calibration_bytes) != (
    "87bfe4072cc4f8b1eb90fe961d4c348552f674223d6d49a85f5e7e0c7e1931e0"
):
    raise RuntimeError("Frozen calibration checksum mismatch.")

seed = 2026111407
rng = np.random.Generator(np.random.PCG64(seed))
selected = rng.choice(len(test["bits"]), size=16, replace=False)
scenarios = []

for number, row_index in enumerate(selected):
    row_index = int(row_index)
    bit = number // 2
    layer_id = int(test["layer"][row_index])
    layer = manifest["inventory"][layer_id]
    channel = int(test["channel"][row_index])

    if channel not in layer["channels"]["test"]:
        raise RuntimeError("Selected channel is outside test partition.")

    unsigned = int(
        np.sum(test["bits"][row_index].astype(np.int64) * (1 << np.arange(8)))
    )
    faulty_unsigned = unsigned ^ (1 << bit)
    scenarios.append({
        "scenario_id": f"S06B-E2E-{number + 1:03d}",
        "dataset_test_row": row_index,
        "layer_id": layer_id,
        "layer": layer["layer"],
        "channel": channel,
        "flat_index": int(test["flat_index"][row_index]),
        "injected_bit": bit,
        "q_clean": unsigned if unsigned < 128 else unsigned - 256,
        "q_faulty": (
            faulty_unsigned if faulty_unsigned < 128 else faulty_unsigned - 256
        ),
    })

protocol = {
    "status": "FROZEN_EXPLORATORY_END_TO_END_PILOT",
    "version": "1.0",
    "seed": seed,
    "selection": "16 distinct test target rows uniformly sampled; 2 per bit",
    "selection_uses_recovery_or_AP_results": False,
    "dataset_sha256": session["dataset_sha256"],
    "manifest_sha256": session["manifest_sha256"],
    "calibration_sha256": sha(calibration_bytes),
    "screening_plan_sha256": sha(plan_bytes),
    "image_ids": plan05b["image_ids"],
    "image_scope": "same_200_development_images_not_held_out",
    "settings": plan05b["settings"],
    "checkpoint_sha256": plan05b["checkpoint_sha256"],
    "clean_model_state_sha256": plan05b["clean_model_state_sha256"],
    "scenarios": scenarios,
    "conditions": [
        "shared_INT8_clean",
        "faulty_no_intervention",
        "faulty_after_C",
        "faulty_after_D",
        "clean_target_after_C",
        "clean_target_after_D"
    ],
    "method_scope": (
        "Inspect only the selected target weight, score all 8 bits, "
        "apply frozen threshold, at most one flip; clean neighbors."
    ),
    "oracle_information": (
        "Clean value and injected bit are evaluator-only; "
        "never inputs to the repair decision."
    ),
    "primary": "paired_AP50_95_after_method_minus_faulty_no_intervention",
    "secondary": [
        "AP50_95_after_clean_target_intervention_minus_shared_clean",
        "all_12_COCOeval_metrics",
        "exact_weight_recovery",
        "method_action_and_selected_bit"
    ],
    "execution": {
        "cpu_supported": True,
        "one_explicit_device_per_session": True,
        "same_environment_and_pipeline_for_all_conditions": True,
        "restore_clean_state_between_conditions": True,
        "reuse_identical_state_predictions": (
            "only_with_verified_state_image_settings_and_environment_identity"
        ),
        "preserve_every_scenario_including_abstentions_and_failures": True
    },
    "limitations": [
        "Exploratory: 06a and 06b test outcomes were previously inspected.",
        "Only two sampled faults per bit; not a precise population estimate.",
        "Target weight under inspection is given; no network-wide localization.",
        "Clean-target controls inspect one weight, not a whole-network scan.",
        "Same development images as 05b; no independent image-test claim.",
        "No threshold tuning or scenario replacement after seeing results.",
        "Logical INT8 weights with FP32 computation, not native INT8."
    ],
    "current_05b_campaign_changed": False
}

content = (json.dumps(protocol, indent=2, allow_nan=False) + "\n").encode()
DEST.parent.mkdir(parents=True, exist_ok=True)
if DEST.exists():
    if DEST.read_bytes() != content:
        raise RuntimeError("Existing protocol differs; nothing overwritten.")
    print("IDENTICAL PROTOCOL ALREADY EXISTS")
else:
    with DEST.open("xb") as stream:
        stream.write(content)
    print("NEW PROTOCOL SAVED")

assert DEST.read_bytes() == content
for item in scenarios:
    print(
        item["scenario_id"], item["layer"],
        f"bit={item['injected_bit']}",
        f"flat={item['flat_index']}",
        f"q={item['q_clean']} -> {item['q_faulty']}"
    )
print("Path:", DEST)
print("SHA256:", sha(content))
print("No model loading, inference, or changes to 05b.")

NEW PROTOCOL SAVED
S06B-E2E-001 model.6.cv1.conv bit=0 flat=12850 q=7 -> 6
S06B-E2E-002 model.22.cv3.0.0.conv bit=0 flat=50404 q=-19 -> -20
S06B-E2E-003 model.22.cv2.2.1.conv bit=1 flat=14735 q=-1 -> -3
S06B-E2E-004 model.15.cv2.conv bit=1 flat=12891 q=29 -> 31
S06B-E2E-005 model.22.cv2.1.1.conv bit=2 flat=7427 q=3 -> 7
S06B-E2E-006 model.2.cv1.conv bit=2 flat=3781 q=6 -> 2
S06B-E2E-007 model.4.m.0.cv1.conv bit=3 flat=16066 q=12 -> 4
S06B-E2E-008 model.8.cv1.conv bit=3 flat=240455 q=-28 -> -20
S06B-E2E-009 model.6.m.1.cv2.conv bit=4 flat=8865 q=-19 -> -3
S06B-E2E-010 model.22.cv2.2.1.conv bit=4 flat=16339 q=-27 -> -11
S06B-E2E-011 model.2.m.0.cv1.conv bit=5 flat=464 q=12 -> 44
S06B-E2E-012 model.18.cv2.conv bit=5 flat=77671 q=3 -> 35
S06B-E2E-013 model.8.m.0.cv2.conv bit=6 flat=498551 q=16 -> 80
S06B-E2E-014 model.3.conv bit=6 flat=50500 q=2 -> 66
S06B-E2E-015 model.22.cv2.0.0.conv bit=7 flat=910 q=-4 -> 124
S06B-E2E-016 model.6.cv2.conv bit=7 flat=105469 q=48 -> -80
Path: /content/dri

In [24]:
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError

protected = ("torch", "torchvision", "numpy", "scikit-learn")
before = {name: version(name) for name in protected}

try:
    installed = version("ultralytics")
except PackageNotFoundError:
    installed = None

if installed is None:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-deps", "ultralytics==8.4.189",
        ],
        check=True,
    )
elif installed != "8.4.189":
    raise RuntimeError(
        f"Unexpected Ultralytics version: {installed}; nothing changed."
    )

after = {name: version(name) for name in protected}
if before != after:
    raise RuntimeError("Protected package versions changed.")

from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import scale_boxes

print("Ultralytics:", version("ultralytics"))
print("Protected package versions unchanged:", after)
print("Required imports: PASS")
print("No checkpoint loaded; no inference or campaign execution.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.189
Protected package versions unchanged: {'torch': '2.14.1+cu130', 'torchvision': '0.29.1+cu130', 'numpy': '2.4.6', 'scikit-learn': '1.6.1'}
Required imports: PASS
No checkpoint loaded; no inference or campaign execution.


In [25]:
from pathlib import Path
import hashlib
import importlib.util
import json
import torch

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
PROJECT = ROOT.parent

protocol_path = (
    ROOT / "experiments/notebook06b/end_to_end_pilot_protocol_v1.json"
)
protocol_bytes = protocol_path.read_bytes()
if hashlib.sha256(protocol_bytes).hexdigest() != (
    "89aa9b56a93fd72c19bceb2e5deff64f06f43e82e47ff1e764ac4e318e3413a8"
):
    raise RuntimeError("06b protocol checksum mismatch.")
protocol_e2e = json.loads(protocol_bytes)

plan_path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
plan_bytes = plan_path.read_bytes()
if hashlib.sha256(plan_bytes).hexdigest() != protocol_e2e["screening_plan_sha256"]:
    raise RuntimeError("05b source plan checksum mismatch.")
plan05b = json.loads(plan_bytes)

module_path = ROOT / "src/screen05b_campaign_v1.py"
if hashlib.sha256(module_path.read_bytes()).hexdigest() != plan05b["source_sha256"]:
    raise RuntimeError("05b source module checksum mismatch.")

spec = importlib.util.spec_from_file_location("screen05b_for_06b", module_path)
screen05b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(screen05b)

# Prepare in memory only; do not call Campaign.run() or create_plan().
torch.set_num_threads(1)
prepared06b = screen05b.Campaign(PROJECT)
prepared06b._load_sources()
prepared06b._model("cpu")

if prepared06b.clean_hash != protocol_e2e["clean_model_state_sha256"]:
    raise RuntimeError("06b clean model state mismatch.")
if prepared06b.ids != protocol_e2e["image_ids"]:
    raise RuntimeError("06b image list mismatch.")

for scenario in protocol_e2e["scenarios"]:
    q, scales = prepared06b.quantized[scenario["layer"]]
    flat = scenario["flat_index"]
    channel = flat // (q.numel() // q.shape[0])
    actual = int(q.flatten()[flat].item())
    faulty = screen05b.bitflip(actual, scenario["injected_bit"])

    if (
        actual != scenario["q_clean"]
        or faulty != scenario["q_faulty"]
        or channel != scenario["channel"]
    ):
        raise RuntimeError(f"Target mismatch: {scenario['scenario_id']}")

    print(
        f"PASS: {scenario['scenario_id']} | "
        f"{scenario['layer']} | q={actual} -> {faulty}"
    )

prepared06b._assert_state(prepared06b.clean_hash)
print("\nClean model state: PASS")
print("CPU model prepared; 16 scenario targets verified.")
print("No COCO image inference, file writes, or 05b campaign execution.")

PASS: S06B-E2E-001 | model.6.cv1.conv | q=7 -> 6
PASS: S06B-E2E-002 | model.22.cv3.0.0.conv | q=-19 -> -20
PASS: S06B-E2E-003 | model.22.cv2.2.1.conv | q=-1 -> -3
PASS: S06B-E2E-004 | model.15.cv2.conv | q=29 -> 31
PASS: S06B-E2E-005 | model.22.cv2.1.1.conv | q=3 -> 7
PASS: S06B-E2E-006 | model.2.cv1.conv | q=6 -> 2
PASS: S06B-E2E-007 | model.4.m.0.cv1.conv | q=12 -> 4
PASS: S06B-E2E-008 | model.8.cv1.conv | q=-28 -> -20
PASS: S06B-E2E-009 | model.6.m.1.cv2.conv | q=-19 -> -3
PASS: S06B-E2E-010 | model.22.cv2.2.1.conv | q=-27 -> -11
PASS: S06B-E2E-011 | model.2.m.0.cv1.conv | q=12 -> 44
PASS: S06B-E2E-012 | model.18.cv2.conv | q=3 -> 35
PASS: S06B-E2E-013 | model.8.m.0.cv2.conv | q=16 -> 80
PASS: S06B-E2E-014 | model.3.conv | q=2 -> 66
PASS: S06B-E2E-015 | model.22.cv2.0.0.conv | q=-4 -> 124
PASS: S06B-E2E-016 | model.6.cv2.conv | q=48 -> -80

Clean model state: PASS
CPU model prepared; 16 scenario targets verified.
No COCO image inference, file writes, or 05b campaign execution.


In [26]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/decisions06b_e2e_v1.py"

source = r'''"""Materialize frozen 06b decisions without fitting or predictor execution."""
import hashlib
import io
import json
from pathlib import Path

PROTOCOL_SHA = "89aa9b56a93fd72c19bceb2e5deff64f06f43e82e47ff1e764ac4e318e3413a8"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def record(path):
    envelope = json.loads(path.read_bytes())
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Record checksum mismatch.")
    return envelope["payload"]

def flip(value, bit):
    unsigned = (int(value) & 255) ^ (1 << int(bit))
    return unsigned - 256 if unsigned >= 128 else unsigned

def run(repo):
    import numpy as np
    repo = Path(repo)
    protocol = json.loads(verified(
        repo / "experiments/notebook06b/end_to_end_pilot_protocol_v1.json",
        PROTOCOL_SHA
    ))
    calibration = json.loads(verified(
        repo / "results/notebook06b/calibration_v1.json",
        protocol["calibration_sha256"]
    ))
    if sha(canonical(calibration["payload"])) != calibration["payload_sha256"]:
        raise RuntimeError("Calibration payload mismatch.")
    calibration = calibration["payload"]

    previous = repo / "results/notebook06b/evaluation_v1"
    summary = record(previous / "summary.json")
    if (
        summary["calibration_sha256"] != protocol["calibration_sha256"]
        or summary["protocol_sha256"] != calibration["protocol_sha256"]
        or summary["source_session_fingerprint"] != calibration["source_session_fingerprint"]
    ):
        raise RuntimeError("Evaluation provenance mismatch.")
    raw = verified(previous / "outcomes.npz", summary["outcomes_sha256"])
    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
        outcomes = {key: archive[key].copy() for key in archive.files}

    dataset = verified(
        repo / "results/notebook06a/masked_bit_dataset_v1/dataset.npz",
        protocol["dataset_sha256"]
    )
    with np.load(io.BytesIO(dataset), allow_pickle=False) as archive:
        test = {key: archive[f"test_{key}"].copy()
                for key in ("bits", "layer", "channel", "flat_index")}

    selected = np.array([s["dataset_test_row"] for s in protocol["scenarios"]])
    clean_scores = {m: np.empty((len(selected), 8)) for m in ("C", "D")}
    source06a = repo / "results/notebook06a/masked_bit_comparison_v1"
    for bit in range(8):
        result = record(source06a / f"bit_{bit}_result.json")
        expected = calibration["prediction_sha256_by_bit"][str(bit)]
        if (
            result["prediction_sha256"] != expected
            or result["session_fingerprint"] != summary["source_session_fingerprint"]
            or result["bit"] != bit
        ):
            raise RuntimeError("Saved predictor output provenance mismatch.")
        raw = verified(source06a / f"bit_{bit}_probabilities.npz", expected)
        with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
            for m in clean_scores:
                p = archive[f"test_{m}"][selected]
                clean_scores[m][:, bit] = np.where(
                    test["bits"][selected, bit] == 0, p, 1-p
                )

    rows = []
    states = {("clean",)}
    for index, scenario in enumerate(protocol["scenarios"]):
        row_index = scenario["dataset_test_row"]
        injected = scenario["injected_bit"]
        for key in ("layer", "channel", "flat_index"):
            expected = scenario["layer_id"] if key == "layer" else scenario[key]
            if int(test[key][row_index]) != expected:
                raise RuntimeError("Selected dataset location mismatch.")
            if int(outcomes[key][row_index]) != expected:
                raise RuntimeError("Selected outcome location mismatch.")

        unsigned = int(np.sum(
            test["bits"][row_index].astype(np.int64) * (1 << np.arange(8))
        ))
        clean = unsigned - 256 if unsigned >= 128 else unsigned
        if clean != scenario["q_clean"] or flip(clean, injected) != scenario["q_faulty"]:
            raise RuntimeError("Selected integer mismatch.")

        row = {"scenario": scenario, "conditions": {}}
        row["conditions"]["shared_INT8_clean"] = {"q_final": clean}
        row["conditions"]["faulty_no_intervention"] = {"q_final": scenario["q_faulty"]}

        for m in ("C", "D"):
            acted = bool(outcomes[f"{m}_fault_intervention"][row_index, injected])
            candidate = int(outcomes[f"{m}_fault_candidate"][row_index, injected])
            final = flip(scenario["q_faulty"], candidate) if acted else scenario["q_faulty"]
            recovered = final == clean
            if recovered != bool(outcomes[f"{m}_exact_recovery"][row_index, injected]):
                raise RuntimeError("Recovery decision mismatch.")
            row["conditions"][f"faulty_after_{m}"] = {
                "acted": acted, "candidate_bit": candidate,
                "q_final": final, "exact_recovery": recovered
            }

            scores = clean_scores[m][index]
            candidate = int(scores.argmax())
            acted = bool(scores[candidate] > calibration["methods"][m]["threshold"])
            if acted != bool(outcomes[f"{m}_clean_intervention"][row_index]):
                raise RuntimeError("Clean decision mismatch.")
            row["conditions"][f"clean_target_after_{m}"] = {
                "acted": acted, "candidate_bit": candidate,
                "q_final": flip(clean, candidate) if acted else clean
            }

        for condition in row["conditions"].values():
            final = condition["q_final"]
            states.add(
                ("clean",) if final == clean else
                (scenario["layer"], scenario["flat_index"], final)
            )
        rows.append(row)

    payload = {
        "status": "FROZEN_DECISIONS_MATERIALIZED_NO_IMAGE_INFERENCE",
        "protocol_sha256": PROTOCOL_SHA,
        "calibration_sha256": protocol["calibration_sha256"],
        "evaluation_summary_payload_sha256": sha(canonical(summary)),
        "outcomes_sha256": summary["outcomes_sha256"],
        "module_sha256": sha(Path(__file__).read_bytes()),
        "unique_weight_states_including_clean": len(states),
        "rows": rows
    }
    content = (json.dumps(
        {"payload_sha256": sha(canonical(payload)), "payload": payload},
        indent=2, allow_nan=False
    ) + "\n").encode()
    out = repo / "results/notebook06b/end_to_end_decisions_v1.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists():
        if out.read_bytes() != content:
            raise RuntimeError("Existing decisions differ; nothing overwritten.")
    else:
        with out.open("xb") as stream:
            stream.write(content)
    if out.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

    for row in rows:
        c = row["conditions"]
        print(
            row["scenario"]["scenario_id"],
            f"C: acted={c['faulty_after_C']['acted']}, recovered={c['faulty_after_C']['exact_recovery']}",
            f"D: acted={c['faulty_after_D']['acted']}, recovered={c['faulty_after_D']['exact_recovery']}"
        )
    for m in ("C", "D"):
        count = sum(r["conditions"][f"clean_target_after_{m}"]["acted"] for r in rows)
        print(f"{m}: clean-target interventions={count}/16")
    print("Unique weight states including clean:", len(states))
    print("Saved:", out)
    print("SHA256:", sha(content))
    print("No model fitting, predictor execution, or image inference.")
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
else:
    with path.open("xb") as stream:
        stream.write(content)

assert path.read_bytes() == content
spec = importlib.util.spec_from_file_location("decisions06b_e2e_v1", path)
decisions06b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(decisions06b)
decisions_e2e = decisions06b.run(ROOT)

S06B-E2E-001 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-002 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-003 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-004 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-005 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-006 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-007 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-008 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-009 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-010 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-011 C: acted=False, recovered=False D: acted=False, recovered=False
S06B-E2E-012 C: acted=True, recovered=True D: acted=False, recovered=False
S06B-E2E-013 C: acted=True, recovered=True D: acted=False, recovered=False
S06

In [27]:
from pathlib import Path
import ast
import hashlib
import json
import textwrap

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
screen_path = ROOT / "src/screen05b_campaign_v1.py"
screen_source = screen_path.read_text(encoding="utf-8")

plan_path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
plan_bytes = plan_path.read_bytes()
plan_sha = json.loads(
    plan_path.with_suffix(".sha256.json").read_bytes()
)["sha256"]
if hashlib.sha256(plan_bytes).hexdigest() != plan_sha:
    raise RuntimeError("Source plan checksum mismatch.")
if hashlib.sha256(screen_path.read_bytes()).hexdigest() != json.loads(
    plan_bytes
)["source_sha256"]:
    raise RuntimeError("Source runner checksum mismatch.")

tree = ast.parse(screen_source)
campaign_class = next(
    node for node in tree.body
    if isinstance(node, ast.ClassDef) and node.name == "Campaign"
)
predict_node = next(
    node for node in campaign_class.body
    if isinstance(node, ast.FunctionDef) and node.name == "_predict"
)
portable = textwrap.dedent(
    "\n".join(screen_source.splitlines()[predict_node.lineno - 1:predict_node.end_lineno])
)
if portable.count("torch.cuda.synchronize()") != 2:
    raise RuntimeError("Unexpected synchronization implementation.")
if portable.count(".to('cuda:0')") != 1:
    raise RuntimeError("Unexpected device implementation.")
portable = portable.replace(
    "def _predict(self):", "def portable_predict(self):", 1
).replace(
    "torch.cuda.synchronize()", "self._synchronize()"
).replace(
    ".to('cuda:0')", ".to(self.device)"
)

body = r'''"""Resumable frozen 06b end-to-end pilot; independent of campaign output."""
import contextlib
import hashlib
import importlib.util
import io
import json
import math
import platform
import time
import types
from pathlib import Path

PROTOCOL_SHA = "89aa9b56a93fd72c19bceb2e5deff64f06f43e82e47ff1e764ac4e318e3413a8"
DECISIONS_SHA = "89a9a3c3e2b4f8187089325a765b1a715b7059d40ed8446fe1f08e3881fab2c0"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

class Pilot:
    def __init__(self, project_root, device="cpu"):
        import torch
        from importlib.metadata import version
        from pycocotools.coco import COCO

        if device not in ("cpu", "cuda:0"):
            raise ValueError("Choose explicitly: cpu or cuda:0.")
        if device == "cuda:0" and not torch.cuda.is_available():
            raise RuntimeError("CUDA unavailable; no silent device change.")
        self.root = Path(project_root)
        self.repo = self.root / "code"
        self.out = self.repo / "results/notebook06b/end_to_end_pilot_v1"
        self.protocol = json.loads(verified(
            self.repo / "experiments/notebook06b/end_to_end_pilot_protocol_v1.json",
            PROTOCOL_SHA
        ))
        envelope = json.loads(verified(
            self.repo / "results/notebook06b/end_to_end_decisions_v1.json",
            DECISIONS_SHA
        ))
        canonical = lambda x: json.dumps(
            x, sort_keys=True, ensure_ascii=False,
            allow_nan=False, separators=(",", ":")
        ).encode()
        self.decisions = envelope["payload"]
        if sha(canonical(self.decisions)) != envelope["payload_sha256"]:
            raise RuntimeError("Decisions payload mismatch.")
        if self.decisions["protocol_sha256"] != PROTOCOL_SHA:
            raise RuntimeError("Decisions protocol mismatch.")
        if [r["scenario"] for r in self.decisions["rows"]] != self.protocol["scenarios"]:
            raise RuntimeError("Decision scenario mismatch.")

        plan = json.loads(verified(
            self.repo / "experiments/notebook05b/int8_screening_plan_v1.json",
            self.protocol["screening_plan_sha256"]
        ))
        source_path = self.repo / "src/screen05b_campaign_v1.py"
        verified(source_path, plan["source_sha256"])
        spec = importlib.util.spec_from_file_location("verified_screen_e2e", source_path)
        screen = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(screen)
        self.c = screen.Campaign(self.root)
        self.c._load_sources()
        torch.set_num_threads(1)
        self.c._model(device)
        if self.c.clean_hash != self.protocol["clean_model_state_sha256"]:
            raise RuntimeError("Clean state mismatch.")
        if self.c.ids != self.protocol["image_ids"]:
            raise RuntimeError("Image IDs mismatch.")
        if self.c.config["settings"] != self.protocol["settings"]:
            raise RuntimeError("Inference settings mismatch.")

        self.c.device = device
        self.c._synchronize = (
            (lambda: torch.cuda.synchronize(device))
            if device == "cuda:0" else (lambda: None)
        )
        self.c._predict = types.MethodType(portable_predict, self.c)
        self.c._cache()
        self.c.category_ids = {r["id"] for r in self.c.categories}
        with contextlib.redirect_stdout(io.StringIO()):
            self.c.gt = COCO(str(self.c.annotation_path))

        self.states = [None]
        self.condition_states = {}
        for row in self.decisions["rows"]:
            scenario = row["scenario"]
            q, _ = self.c.quantized[scenario["layer"]]
            actual = int(q.flatten()[scenario["flat_index"]].item())
            if actual != scenario["q_clean"]:
                raise RuntimeError("Clean target mismatch.")
            for condition, action in row["conditions"].items():
                final = action["q_final"]
                if type(final) is not int or not -128 <= final <= 127:
                    raise RuntimeError("Invalid final integer.")
                state = None if final == actual else (
                    scenario["layer"], scenario["flat_index"], final
                )
                if state not in self.states:
                    self.states.append(state)
                self.condition_states[
                    scenario["scenario_id"] + "/" + condition
                ] = self.states.index(state)
        if len(self.states) != self.decisions["unique_weight_states_including_clean"]:
            raise RuntimeError("Unique state count mismatch.")

        self.session = {
            "protocol_sha256": PROTOCOL_SHA,
            "decisions_sha256": DECISIONS_SHA,
            "module_sha256": sha(Path(__file__).read_bytes()),
            "screen_source_sha256": plan["source_sha256"],
            "device": device,
            "python": platform.python_version(),
            "packages": {p: version(p) for p in (
                "torch", "torchvision", "ultralytics", "numpy",
                "pycocotools", "opencv-python", "Pillow"
            )},
            "gpu": torch.cuda.get_device_name(device) if device == "cuda:0" else None,
            "threads": torch.get_num_threads(),
            "tf32": False,
            "deterministic_algorithms": True,
            "image_ids": self.c.ids,
            "image_sha256": self.c.image_hashes,
            "clean_state": self.c.clean_hash,
            "states": [list(s) if s is not None else None for s in self.states],
            "condition_states": self.condition_states
        }
        self.fingerprint = sha(canonical(self.session))
        self.c._assert_state(self.c.clean_hash)
        print("Prepared:", device, "| unique states:", len(self.states))
        print("No COCO image inference yet.")

    def _read(self, path):
        record = self.c.base.read_record(path)
        if record["fingerprint"] != self.fingerprint:
            raise RuntimeError("Stored execution fingerprint mismatch.")
        return record

    def _execute(self, index):
        import torch
        state = self.states[index]
        self.c._assert_state(self.c.clean_hash)
        target = original = None
        try:
            if state is not None:
                layer, flat, final = state
                target = self.c.modules[layer].weight
                original = target.detach().clone()
                q, scales = self.c.quantized[layer]
                channel = flat // (q.numel() // q.shape[0])
                value = (
                    torch.tensor(final, dtype=torch.float32)
                    * scales.flatten()[channel]
                ).item()
                with torch.no_grad():
                    target.flatten()[flat] = value
                changed = (target != original).flatten().nonzero().flatten().tolist()
                if changed != [flat]:
                    raise RuntimeError("Single-weight isolation failed.")
                with torch.no_grad():
                    target.flatten()[flat] = original.flatten()[flat]
                self.c._assert_state(self.c.clean_hash)
                with torch.no_grad():
                    target.flatten()[flat] = value

            expected_state = self.c._state()
            prediction_path = self.out / f"state_{index:03d}_predictions.json.gz"
            if prediction_path.exists():
                saved = self._read(prediction_path)
                if saved["index"] != index or saved["state_sha256"] != expected_state:
                    raise RuntimeError("Saved prediction state mismatch.")
                predictions = saved["data"]
            else:
                predictions = self.c._predict()
                self.c._assert_state(expected_state)
                self.c._validate_predictions(predictions)
                self.c.base.write_record(prediction_path, {
                    "fingerprint": self.fingerprint, "index": index,
                    "state_sha256": expected_state, "data": predictions
                })
            self.c._validate_predictions(predictions)
            self.c._assert_state(expected_state)
        finally:
            if original is not None:
                with torch.no_grad():
                    target.copy_(original)
            self.c._assert_state(self.c.clean_hash)

        metrics, seconds = self.c._evaluate(predictions["predictions"])
        if any(not math.isfinite(v) for v in metrics.values()) or metrics["AP50_95"] < 0:
            raise RuntimeError("Invalid COCO metrics.")
        result = {
            "fingerprint": self.fingerprint, "index": index,
            "state": list(state) if state is not None else None,
            "state_sha256": expected_state,
            "prediction_sha256": sha(prediction_path.read_bytes()),
            "metrics": metrics, "evaluation_seconds": seconds,
            "exact_clean_restoration": True
        }
        self.c.base.write_record(
            self.out / f"state_{index:03d}_evaluation.json.gz", result
        )
        return result

    def run(self, max_new_states=1):
        import fcntl
        if type(max_new_states) is not int or max_new_states < 0:
            raise ValueError("Use a nonnegative integer state limit.")
        self.out.mkdir(parents=True, exist_ok=True)
        with (self.out / "writer.lock").open("a") as lock:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
            if list(self.out.glob("*.part")) or list(self.out.glob("*_failure.json.gz")):
                raise RuntimeError("Partial/failure records require review.")
            session_path = self.out / "session.json.gz"
            if session_path.exists():
                if self.c.base.read_record(session_path) != self.session:
                    raise RuntimeError("Resume session differs; no overwrite.")
            else:
                self.c.base.write_record(session_path, self.session)

            completed, new = 0, 0
            for index in range(len(self.states)):
                path = self.out / f"state_{index:03d}_evaluation.json.gz"
                if path.exists():
                    result = self._read(path)
                    expected = list(self.states[index]) if self.states[index] is not None else None
                    if result["index"] != index or result["state"] != expected:
                        raise RuntimeError("Saved evaluation state mismatch.")
                    prediction_path = self.out / f"state_{index:03d}_predictions.json.gz"
                    if result["prediction_sha256"] != sha(prediction_path.read_bytes()):
                        raise RuntimeError("Saved prediction checksum mismatch.")
                    prediction = self._read(prediction_path)
                    if prediction["state_sha256"] != result["state_sha256"]:
                        raise RuntimeError("Prediction/evaluation state mismatch.")
                    self.c._validate_predictions(prediction["data"])
                    completed += 1
                    continue
                if new >= max_new_states:
                    break
                try:
                    result = self._execute(index)
                except BaseException as exc:
                    failure = self.out / f"state_{index:03d}_failure.json.gz"
                    if not failure.exists():
                        self.c.base.write_record(failure, {
                            "fingerprint": self.fingerprint, "index": index,
                            "exception": type(exc).__name__, "message": str(exc),
                            "clean_state_restored": self.c._state() == self.c.clean_hash
                        })
                    raise
                new += 1
                completed += 1
                print(
                    f"State {index:03d}: AP50:95={result['metrics']['AP50_95']:.9f} | saved",
                    flush=True
                )
            self.c._assert_state(self.c.clean_hash)
            print(f"Completed states: {completed}/{len(self.states)} | new: {new}")
            print("Output:", self.out)
            print("05b campaign output unchanged.")
            return {"completed_states": completed, "new_states": new}
'''

source = body + "\n\n" + portable + "\n"
path = ROOT / "src/end_to_end06b_v1.py"
compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
    print("IDENTICAL MODULE ALREADY EXISTS")
else:
    with path.open("xb") as stream:
        stream.write(content)
    print("NEW MODULE SAVED")

assert path.read_bytes() == content
print("Path:", path)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Syntax: PASS")
print("No image inference or campaign execution.")

NEW MODULE SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/src/end_to_end06b_v1.py
SHA256: 29527d800e61ff598a813ebd425194963b81b7da1aa4f64ab24a942e818d3ac8
Syntax: PASS
No image inference or campaign execution.


In [28]:
from pathlib import Path
import hashlib
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/end_to_end06b_v1.py"

if hashlib.sha256(path.read_bytes()).hexdigest() != (
    "29527d800e61ff598a813ebd425194963b81b7da1aa4f64ab24a942e818d3ac8"
):
    raise RuntimeError("End-to-end module checksum mismatch.")

spec = importlib.util.spec_from_file_location("end_to_end06b_v1", path)
end_to_end06b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(end_to_end06b)

pilot06b = end_to_end06b.Pilot(ROOT.parent, device="cuda:0")
baseline_status = pilot06b.run(max_new_states=1)

Prepared: cuda:0 | unique states: 17
No COCO image inference yet.
State 000: AP50:95=0.484276959 | saved
Completed states: 1/17 | new: 1
Output: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook06b/end_to_end_pilot_v1
05b campaign output unchanged.


In [29]:
first_fault_status = pilot06b.run(max_new_states=1)

State 001: AP50:95=0.484276745 | saved
Completed states: 2/17 | new: 1
Output: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook06b/end_to_end_pilot_v1
05b campaign output unchanged.


In [30]:
remaining_status = pilot06b.run(max_new_states=15)

State 002: AP50:95=0.484276958 | saved
State 003: AP50:95=0.484276959 | saved
State 004: AP50:95=0.484330751 | saved
State 005: AP50:95=0.484277198 | saved
State 006: AP50:95=0.483434818 | saved
State 007: AP50:95=0.484283669 | saved
State 008: AP50:95=0.484276396 | saved
State 009: AP50:95=0.484210860 | saved
State 010: AP50:95=0.484312396 | saved
State 011: AP50:95=0.478452249 | saved
State 012: AP50:95=0.483965737 | saved
State 013: AP50:95=0.484282112 | saved
State 014: AP50:95=0.484390958 | saved
State 015: AP50:95=0.484402906 | saved
State 016: AP50:95=0.482926162 | saved
Completed states: 17/17 | new: 15
Output: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook06b/end_to_end_pilot_v1
05b campaign output unchanged.


In [31]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/report06b_e2e_v1.py"

source = r'''"""Verify saved end-to-end records and report all frozen conditions."""
import csv
import gzip
import hashlib
import io
import json
from pathlib import Path

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def unpack(content):
    envelope = json.loads(content)
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Record payload checksum mismatch.")
    return envelope["payload"]

def read_gzip(path):
    return unpack(gzip.decompress(path.read_bytes()))

def publish(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes() != content:
            raise RuntimeError(f"Existing file differs: {path.name}")
    else:
        with path.open("xb") as stream:
            stream.write(content)
    if path.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

def run(repo):
    repo = Path(repo)
    folder = repo / "results/notebook06b/end_to_end_pilot_v1"
    session = read_gzip(folder / "session.json.gz")
    fingerprint = sha(canonical(session))

    decision_path = repo / "results/notebook06b/end_to_end_decisions_v1.json"
    decision_bytes = decision_path.read_bytes()
    if sha(decision_bytes) != session["decisions_sha256"]:
        raise RuntimeError("Decisions checksum mismatch.")
    decisions = unpack(decision_bytes)

    evaluations = {}
    hashes = {}
    for index, state in enumerate(session["states"]):
        evaluation_path = folder / f"state_{index:03d}_evaluation.json.gz"
        record = read_gzip(evaluation_path)
        prediction_path = folder / f"state_{index:03d}_predictions.json.gz"
        prediction = read_gzip(prediction_path)
        if (
            record["fingerprint"] != fingerprint
            or prediction["fingerprint"] != fingerprint
            or record["index"] != index
            or prediction["index"] != index
            or record["state"] != state
            or record["state_sha256"] != prediction["state_sha256"]
            or record["prediction_sha256"] != sha(prediction_path.read_bytes())
            or prediction["data"]["image_ids"] != session["image_ids"]
            or record["exact_clean_restoration"] is not True
        ):
            raise RuntimeError(f"State provenance mismatch: {index}")
        evaluations[index] = record
        hashes[str(index)] = sha(evaluation_path.read_bytes())

    rows, detailed = [], []
    for item in decisions["rows"]:
        scenario = item["scenario"]
        sid = scenario["scenario_id"]
        metrics = {}
        for condition, action in item["conditions"].items():
            state_index = session["condition_states"][sid + "/" + condition]
            expected_state = (
                None if action["q_final"] == scenario["q_clean"] else
                [scenario["layer"], scenario["flat_index"], action["q_final"]]
            )
            if session["states"][state_index] != expected_state:
                raise RuntimeError("Condition-to-state mapping mismatch.")
            metrics[condition] = evaluations[state_index]["metrics"]
            detailed.append({
                "scenario_id": sid,
                "condition": condition,
                "state_index": state_index,
                "q_final": action["q_final"],
                **metrics[condition]
            })

        clean = metrics["shared_INT8_clean"]["AP50_95"]
        faulty = metrics["faulty_no_intervention"]["AP50_95"]
        row = {
            "scenario_id": sid,
            "layer": scenario["layer"],
            "bit": scenario["injected_bit"],
            "clean_AP50_95": clean,
            "faulty_AP50_95": faulty,
            "fault_minus_clean_pp": 100 * (faulty - clean),
        }
        for method in ("C", "D"):
            after = metrics[f"faulty_after_{method}"]["AP50_95"]
            clean_after = metrics[f"clean_target_after_{method}"]["AP50_95"]
            row[f"{method}_AP50_95"] = after
            row[f"{method}_gain_vs_faulty_pp"] = 100 * (after - faulty)
            row[f"{method}_change_vs_clean_pp"] = 100 * (after - clean)
            row[f"{method}_clean_control_change_pp"] = 100 * (clean_after - clean)
            row[f"{method}_exact_weight_recovery"] = item["conditions"][
                f"faulty_after_{method}"
            ]["exact_recovery"]
        rows.append(row)

    aggregates = {}
    for method in ("C", "D"):
        gains = [r[f"{method}_gain_vs_faulty_pp"] for r in rows]
        aggregates[method] = {
            "mean_signed_AP_gain_pp": sum(gains) / len(gains),
            "positive_gain_scenarios": sum(x > 0 for x in gains),
            "zero_gain_scenarios": sum(x == 0 for x in gains),
            "negative_gain_scenarios": sum(x < 0 for x in gains),
            "exact_weight_recoveries": sum(
                r[f"{method}_exact_weight_recovery"] for r in rows
            )
        }

    payload = {
        "status": "EXPLORATORY_END_TO_END_PILOT_COMPLETE",
        "session_fingerprint": fingerprint,
        "session_file_sha256": sha((folder / "session.json.gz").read_bytes()),
        "evaluation_sha256_by_state": hashes,
        "report_module_sha256": sha(Path(__file__).read_bytes()),
        "scenario_count": len(rows),
        "unique_states": len(evaluations),
        "aggregates": aggregates,
        "rows": rows,
        "limitations": [
            "16 exploratory scenarios, two per bit; no population efficacy claim.",
            "Selected target weight is given; no network-wide localization.",
            "Development images; previous test outcomes inspected.",
            "Exact weight restoration can reduce AP when the fault increased sample AP.",
            "No clean-target interventions in these 16 controls does not imply zero false alarms.",
            "05b screening remains separate and unchanged."
        ]
    }
    destination = repo / "docs/research_reports/notebook06b_end_to_end_v1"
    for name, records in (
        ("scenario_comparison_v1.csv", rows),
        ("all_conditions_cocoeval_v1.csv", detailed),
    ):
        stream = io.StringIO(newline="")
        writer = csv.DictWriter(stream, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
        publish(destination / name, stream.getvalue().encode())

    content = (json.dumps(
        {"payload_sha256": sha(canonical(payload)), "payload": payload},
        indent=2, allow_nan=False
    ) + "\n").encode()
    publish(destination / "summary_v1.json", content)

    print("Verified states:", len(evaluations))
    print("Scenario | fault-clean pp | C gain pp | D gain pp")
    for row in rows:
        print(
            f"{row['scenario_id']} | {row['fault_minus_clean_pp']:+.8f} | "
            f"{row['C_gain_vs_faulty_pp']:+.8f} | "
            f"{row['D_gain_vs_faulty_pp']:+.8f}"
        )
    print(json.dumps(aggregates, indent=2))
    print("Saved:", destination)
    print("Summary SHA256:", sha(content))
    print("No inference, threshold changes, or 05b execution.")
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
else:
    with path.open("xb") as stream:
        stream.write(content)

assert path.read_bytes() == content
spec = importlib.util.spec_from_file_location("report06b_e2e_v1", path)
report06b = importlib.util.module_from_spec(spec)
spec.loader.exec_module(report06b)
end_to_end_report = report06b.run(ROOT)

Verified states: 17
Scenario | fault-clean pp | C gain pp | D gain pp
S06B-E2E-001 | -0.00002145 | +0.00000000 | +0.00000000
S06B-E2E-002 | -0.00000014 | +0.00000000 | +0.00000000
S06B-E2E-003 | +0.00000000 | +0.00000000 | +0.00000000
S06B-E2E-004 | +0.00537913 | +0.00000000 | +0.00000000
S06B-E2E-005 | +0.00002390 | +0.00000000 | +0.00000000
S06B-E2E-006 | -0.08421411 | +0.00000000 | +0.00000000
S06B-E2E-007 | +0.00067102 | +0.00000000 | +0.00000000
S06B-E2E-008 | -0.00005633 | +0.00000000 | +0.00000000
S06B-E2E-009 | -0.00660996 | +0.00000000 | +0.00000000
S06B-E2E-010 | +0.00354369 | +0.00000000 | +0.00000000
S06B-E2E-011 | -0.58247106 | +0.00000000 | +0.00000000
S06B-E2E-012 | -0.03112222 | +0.03112222 | +0.00000000
S06B-E2E-013 | +0.00051528 | -0.00051528 | +0.00000000
S06B-E2E-014 | +0.01139989 | -0.01139989 | -0.01139989
S06B-E2E-015 | +0.01259463 | -0.01259463 | -0.01259463
S06B-E2E-016 | -0.13507974 | +0.00000000 | +0.00000000
{
  "C": {
    "mean_signed_AP_gain_pp": 0.0004132

In [32]:
from pathlib import Path
from importlib.metadata import version
import platform
import torch

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
folder = ROOT / "results/notebook05b/int8_screening_v1"
session_path = folder / "session.json.gz"

if not session_path.is_file():
    raise FileNotFoundError("Previous 05b session not found.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable; campaign not started.")

# Reuse the verified reader and source identities from the prepared pilot.
base = pilot06b.c.base
previous = base.read_record(session_path)

environment = {
    "python": platform.python_version(),
    "packages": {
        package: version(package)
        for package in (
            "torch", "torchvision", "ultralytics", "numpy",
            "pycocotools", "opencv-python", "Pillow"
        )
    },
    "gpu": torch.cuda.get_device_name(0),
    "torch_threads": torch.get_num_threads(),
    "torch_cuda_build": torch.version.cuda,
    "tf32": False,
    "deterministic_algorithms": True,
}

current = {
    "plan_sha256": pilot06b.protocol["screening_plan_sha256"],
    "source_sha256": pilot06b.c.source_sha,
    "environment": environment,
    "image_sha256": pilot06b.c.image_hashes,
    "clean_state": pilot06b.c.clean_hash,
}

print("Session comparisons:")
for key in current:
    print(f"{key}: {'MATCH' if previous.get(key) == current[key] else 'DIFFERENT'}")

print("\nEnvironment differences:")
old_environment = previous.get("environment", {})
for key in sorted(set(old_environment) | set(environment)):
    if old_environment.get(key) != environment.get(key):
        print(
            f"{key}: previous={old_environment.get(key)!r} "
            f"| current={environment.get(key)!r}"
        )

partials = sorted(folder.rglob("*.part"))
failures = sorted(folder.glob("*_failure.json.gz"))
evaluations = list(folder.glob("S05B-*_evaluation.json.gz"))

print("\nSaved scenario evaluations:", len(evaluations))
print("Partial files:", len(partials))
print("Failure records:", len(failures))
for path in partials + failures:
    print(path.relative_to(folder))

runtime_flags_ok = (
    torch.are_deterministic_algorithms_enabled()
    and not torch.backends.cudnn.benchmark
    and not torch.backends.cuda.matmul.allow_tf32
    and not torch.backends.cudnn.allow_tf32
)
resume05b_ok = (
    previous == current
    and not partials
    and not failures
    and runtime_flags_ok
)

print("\nRuntime flags match required settings:", runtime_flags_ok)
print("Preliminary resume compatibility:", "PASS" if resume05b_ok else "REVIEW REQUIRED")
print("Saved scenario artifacts have not yet been fully reverified.")
print("No campaign execution or file changes.")

Session comparisons:
plan_sha256: MATCH
source_sha256: MATCH
environment: MATCH
image_sha256: MATCH
clean_state: MATCH

Environment differences:

Saved scenario evaluations: 1579
Partial files: 0
Failure records: 0

Runtime flags match required settings: True
Preliminary resume compatibility: PASS
Saved scenario artifacts have not yet been fully reverified.
No campaign execution or file changes.


In [33]:
from pathlib import Path
import hashlib
import importlib.util
import json

if not resume05b_ok:
    raise RuntimeError("Resume compatibility has not passed.")

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
plan_path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
plan_bytes = plan_path.read_bytes()

if hashlib.sha256(plan_bytes).hexdigest() != (
    pilot06b.protocol["screening_plan_sha256"]
):
    raise RuntimeError("Screening plan changed.")

source_path = ROOT / "src/screen05b_campaign_v1.py"
if hashlib.sha256(source_path.read_bytes()).hexdigest() != (
    json.loads(plan_bytes)["source_sha256"]
):
    raise RuntimeError("Campaign source changed.")

# Load a fresh, unmodified campaign instance.
spec = importlib.util.spec_from_file_location("screen05b_resume", source_path)
screen05b_resume = importlib.util.module_from_spec(spec)
spec.loader.exec_module(screen05b_resume)

campaign05b = screen05b_resume.Campaign(ROOT.parent)
progress05b = campaign05b.run(
    max_new_scenarios=25,
    max_wall_seconds=1800,
)

1580/5040 | S05B-01580 | AP change=-0.00052133 | saved
1581/5040 | S05B-01581 | AP change=+0.00003295 | saved
1582/5040 | S05B-01582 | AP change=-0.00058108 | saved
1583/5040 | S05B-01583 | AP change=-0.00035556 | saved
1584/5040 | S05B-01584 | AP change=-0.00347420 | saved
1585/5040 | S05B-01585 | AP change=+0.00000002 | saved
1586/5040 | S05B-01586 | AP change=-0.00000122 | saved
1587/5040 | S05B-01587 | AP change=+0.00000539 | saved
1588/5040 | S05B-01588 | AP change=-0.00000028 | saved
1589/5040 | S05B-01589 | AP change=-0.00020580 | saved
1590/5040 | S05B-01590 | AP change=+0.00004209 | saved
1591/5040 | S05B-01591 | AP change=+0.00008778 | saved
1592/5040 | S05B-01592 | AP change=-0.00000019 | saved
1593/5040 | S05B-01593 | AP change=+0.00000653 | saved
1594/5040 | S05B-01594 | AP change=+0.00000000 | saved
1595/5040 | S05B-01595 | AP change=+0.00002522 | saved
1596/5040 | S05B-01596 | AP change=-0.00006195 | saved
1597/5040 | S05B-01597 | AP change=-0.00000024 | saved
1598/5040 

In [34]:
progress05b = campaign05b.run(
    max_new_scenarios=100,
    max_wall_seconds=3600,
)

1605/5040 | S05B-01605 | AP change=+0.00096168 | saved
1606/5040 | S05B-01606 | AP change=-0.00001796 | saved
1607/5040 | S05B-01607 | AP change=+0.00000014 | saved
1608/5040 | S05B-01608 | AP change=+0.00003016 | saved
1609/5040 | S05B-01609 | AP change=-0.00157146 | saved
1610/5040 | S05B-01610 | AP change=-0.00075045 | saved
1611/5040 | S05B-01611 | AP change=-0.00176402 | saved
1612/5040 | S05B-01612 | AP change=+0.00000000 | saved
1613/5040 | S05B-01613 | AP change=+0.00015756 | saved
1614/5040 | S05B-01614 | AP change=+0.00000002 | saved
1615/5040 | S05B-01615 | AP change=+0.00005075 | saved
1616/5040 | S05B-01616 | AP change=+0.00001923 | saved
1617/5040 | S05B-01617 | AP change=-0.00006816 | saved
1618/5040 | S05B-01618 | AP change=+0.00066517 | saved
1619/5040 | S05B-01619 | AP change=-0.00093518 | saved
1620/5040 | S05B-01620 | AP change=-0.00000129 | saved
1621/5040 | S05B-01621 | AP change=-0.00039391 | saved
1622/5040 | S05B-01622 | AP change=+0.00000000 | saved
1623/5040 

In [1]:
from google.colab import drive
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import sys
import torch

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
torch.set_num_threads(1)

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print("Repository exists:", ROOT.is_dir())

for package in ("torch", "numpy", "scikit-learn", "ultralytics"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

for relative in (
    "src/contextual06a_runner_v1.py",
    "results/notebook06a/masked_bit_dataset_v1/manifest.json",
    "experiments/notebook05b/int8_screening_plan_v1.json",
    "docs/research_reports/notebook05b_screening_v1/progress_01704.json",
):
    print(relative, "FOUND" if (ROOT / relative).is_file() else "MISSING")

print("No model loading, inference, installation, or file changes.")

Mounted at /content/drive
Python: 3.13.18
CUDA available: False
Repository exists: True
torch: 2.14.1+cpu
numpy: 2.4.6
scikit-learn: 1.6.1
ultralytics: NOT INSTALLED
src/contextual06a_runner_v1.py FOUND
results/notebook06a/masked_bit_dataset_v1/manifest.json FOUND
experiments/notebook05b/int8_screening_plan_v1.json FOUND
docs/research_reports/notebook05b_screening_v1/progress_01704.json FOUND
No model loading, inference, installation, or file changes.


In [2]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import math

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
content = path.read_bytes()

expected = json.loads(
    path.with_suffix(".sha256.json").read_bytes()
)["sha256"]
if hashlib.sha256(content).hexdigest() != expected:
    raise RuntimeError("Screening plan checksum mismatch.")

plan = json.loads(content)
inventory = plan["inventory"]
kernel_counts = Counter()
total_weights = 0

for layer in inventory:
    shape = layer["weight_shape"]
    if len(shape) != 4 or math.prod(shape) != layer["weight_count"]:
        raise RuntimeError(f"Invalid weight shape: {layer['name']}")
    kernel_counts[tuple(shape[2:])] += 1
    total_weights += layer["weight_count"]

print("Eligible Conv2d tensors:", len(inventory))
print("Kernel shapes and layer counts:", dict(kernel_counts))
print("Total eligible weights:", total_weights)
print("Logical INT8 weight storage, bytes:", total_weights)

print("\nLayers with kernels other than 1x1 or 3x3:")
for layer in inventory:
    if tuple(layer["weight_shape"][2:]) not in ((1, 1), (3, 3)):
        print(layer["name"], layer["weight_shape"])

print("\nNo model loading, inference, or file changes.")

Eligible Conv2d tensors: 63
Kernel shapes and layer counts: {(3, 3): 39, (1, 1): 24}
Total eligible weights: 11146080
Logical INT8 weight storage, bytes: 11146080

Layers with kernels other than 1x1 or 3x3:

No model loading, inference, or file changes.
